In [ ]:
#!pip install --upgrade transformers

In [ ]:
#import random


In [ ]:
#import numpy as np #eu devo fazer as operações com os tensores (embeddings) usando torch
import numpy as np
import torch
import torch.nn.functional as F
import random
import time

from transformers import BertTokenizer # Or AutoTokenizer
from transformers import BertForPreTraining  # BertForPreTraining for loading pretraining heads; or AutoModelForPreTraining
from transformers import BertModel  # BertModel for BERT without pretraining heads, or AutoModel

#DESCOMENTAR!
model = BertModel.from_pretrained('neuralmind/bert-base-portuguese-cased')
model.to("cuda")
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased', do_lower_case=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Funções auxiliares

In [ ]:
def tokens_embeddings(frase, tokenizer, model):
  '''retorna os tokens e os embeddings correspondentes a uma frase.
  Usa GPU'''

  inputs = tokenizer(frase, return_tensors='pt') #pt é de pytorch
  inputs.to("cuda")
  with torch.no_grad():
    outputs = model(**inputs)



  # Vetores de embeddings
  embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
  tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

  #tokens_limpo1 = tokens[1:len(embeddings)-1]
  #^pra retornar sem [CLS] e [SEP]

  #Aqui seria a parte de tirar as stopwords da lista de tokens

  #return tokens_limpo1


  return (tokens, embeddings)


def similaridade(emb1, emb2):
  '''retorna a similaridade de cosseno entre dois embeddings
  Funciona por causa do
  import torch.nn.functional as F'''

  return (F.cosine_similarity(emb1, emb2, dim=0)).item() #dentro do parenteses é um tensor


def get_stopwords(stopwords_file):
  '''retorna um conjunto com todas as stopwords presentes
  no arquivo txt stopwords_file'''


  arq_stop = open(stopwords_file, "r")
  set_stopwords = set()


  for line in arq_stop: #o txt tem uma palavra por linha
    word = line.replace("\n", "")

    set_stopwords.add(word)

  arq_stop.close()

  return set_stopwords


def similarity_all_pairs(tensores):
  '''Tentativa de otimizar o cálculo da similaridade entre todos os pares de embeddings'''

  # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = tensores.norm(dim=1, keepdim=True)

  # terminando de normalizar
  data_normalized = tensores / norms

  # Calculando a similaridade de cosseno (all pairs)
  similarity_matrix = torch.matmul(data_normalized, data_normalized.T)

  return similarity_matrix


def normaliza(matriz):
  '''retorna a matriz normalizada para o cálculo da similaridade
  all pairs'''

  # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = matriz.norm(dim=1, keepdim=True) #pegando a norma

  # terminando de normalizar
  data_normalized = matriz / norms


  return data_normalized





def gera_frase(corpus):
  lista_palavras = corpus.split()

  tam = random.randint(2,30)

  frase = ""

  for i in range(tam):
    indice = random.randint(0,len(lista_palavras)-1)
    palavra = lista_palavras[indice]
    frase += palavra

    if(i<(tam-1)):
      frase+=" "


  return frase




In [ ]:
#input_ids = tokenizer.encode('Tinha uma pedra no meio do caminho.', return_tensors='pt')



frase1 = "ontem choveu muito"
frase2 = "amanhã certamente vai chover"




tokens1 = tokenizer.tokenize(frase1)
tokens2 = tokenizer.tokenize(frase2)

#OBS: ACHO que a melhor coisa a se fazer quando o BERT quebrar  as
#palavras é: Salvar o embedding que corresponde à MÉDIA desses embeddings;
#Salvar a palavra que corresponde à palavra original
#Se duas palavras iguais tiverem mesma média de embedding nessa tokenização doida,
#ou média razoavelmente parecida, eu junto os nós.
#(Se for stopword descarto)
#DOIS PROBLEMAS:
#1. Detectar que a palavra foi quebrada (tem o ## antes)
#2. Calcular a similaridade usando a média dos embeddings
#OBS!: Sinceramente talvez não seja preciso fazer isso das médias
#os tokens mesmo quebrados vão reter a sua informação contextual
#E isso não vai afetar tanto a coesão no final das contas...
tokens1


['on', '##tem', 'cho', '##veu', 'muito']

In [ ]:
tokens2

['aman', '##ha', 'certamente', 'vai', 'cho', '##ver']

In [ ]:
inputs1 = tokenizer(frase1, return_tensors='pt') #pt é de pytorch
with torch.no_grad():
  outputs1 = model(**inputs1)

# Vetores de embeddings
embeddings1 = outputs1.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
tokens1 = tokenizer.convert_ids_to_tokens(inputs1['input_ids'][0])


In [ ]:
embeddings1[0]

tensor([ 1.7487e-01, -3.4689e-01,  3.7880e-01,  6.9282e-02,  5.1747e-01,
         4.4411e-01,  4.3340e-02,  2.2930e-01,  1.9339e-01,  4.6603e-01,
         1.0683e-01,  4.9301e-01,  4.1468e-01, -4.7703e-01, -1.4891e-02,
         1.5629e-02,  2.4440e-01, -4.6617e-01,  1.7341e-01,  5.4694e-01,
        -4.0005e-02,  1.1537e-01,  2.7760e-01,  4.8419e-01,  2.8925e-01,
         1.2571e-01, -2.9855e-01,  6.4210e-01,  3.3258e-01, -5.6442e-01,
         1.0632e-01,  6.4834e-01,  1.4474e-01,  2.5542e-01, -2.9212e-01,
        -2.1334e-02,  2.3617e-01,  1.8625e-01, -5.1608e-02,  2.2124e-03,
         3.2289e-02,  7.8824e-02, -2.5924e-01, -4.1582e-01, -4.6300e-03,
         3.1577e-02,  1.8644e-01, -9.2398e-02, -5.2648e-01,  2.5955e-02,
        -3.1461e-01, -2.2864e-01, -8.5263e-02, -2.3621e-01,  2.6088e-01,
         6.5643e-01, -3.7354e-02, -2.5983e-01, -9.0652e-02, -2.0670e-01,
         1.2018e-01, -6.8392e-02, -1.4405e-01, -2.7829e-01,  1.9172e-01,
        -2.1276e-01,  4.7113e-01, -3.8418e-01, -3.1

In [ ]:
tokens1

['[CLS]', 'on', '##tem', 'cho', '##veu', 'muito', '[SEP]']

In [ ]:
inputs2 = tokenizer(frase2, return_tensors='pt')
with torch.no_grad():
  outputs2 = model(**inputs2)

# Vetores de embeddings
embeddings2 = outputs2.last_hidden_state.squeeze(0)  # (seq_len, hidden_size)
tokens2 = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])

In [ ]:
len(embeddings2)

8

In [ ]:
tokens2

['[CLS]', 'aman', '##hã', 'certamente', 'vai', 'cho', '##ver', '[SEP]']

In [ ]:
embeddings2[5].dim()

1

In [ ]:
embeddings1[3].dim()

1

In [ ]:
similarity = F.cosine_similarity(embeddings1[3], embeddings2[5], dim=0)
similarity

tensor(0.8943)

In [ ]:
#Falta ver se faz sentido usar a média!!!!
chover = (embeddings2[5] + embeddings2[6])/2


In [ ]:
choveu =(embeddings1[3] + embeddings1[4])/2

In [ ]:
similarity = F.cosine_similarity(chover, choveu, dim=0)
similarity

tensor(0.8154)

In [ ]:
chover

tensor(-0.0051)

In [ ]:
choveu

tensor(-0.0072)

In [ ]:
embeddings2[5] + embeddings2[6]

tensor([-1.2424e-01,  3.7874e-03,  5.2788e-03, -5.6732e-02,  1.1413e+00,
         1.0088e+00, -9.6184e-01, -2.8276e-01,  4.2032e-01,  1.0283e-01,
         1.2482e+00,  9.6490e-01, -2.0544e-01, -6.3037e-01,  1.5993e-01,
        -2.4208e-01,  1.0094e+00,  1.3541e-01,  3.1441e-01,  6.7468e-01,
        -1.3764e-01, -2.8166e-01, -1.4795e+00, -4.4114e-01, -6.0682e-01,
        -6.7433e-01,  6.5603e-01, -1.0727e-01, -1.2627e+00, -1.3025e+00,
         3.3208e-02,  8.0782e-01, -4.6020e-01,  8.1058e-01, -1.0661e-01,
         1.7565e-01,  1.8005e-01,  3.1706e-01,  3.8813e-01, -4.0842e-01,
        -2.6831e-02,  5.8074e-01, -1.0347e+00, -1.2494e+00,  1.4916e-01,
         9.0111e-01, -1.3828e-01, -7.7920e-01, -1.2758e+00, -8.2408e-01,
        -5.4827e-01,  2.3297e-01, -1.5705e+00,  6.4685e-01, -7.4146e-01,
         8.4232e-01, -2.5557e-01, -2.8395e-01,  3.6931e-01, -4.4966e-01,
        -2.1367e-01, -1.2936e+00,  2.7967e-01,  9.7453e-02, -1.3100e+00,
         1.9750e-01, -1.4612e-01, -6.3724e-01, -7.3

In [ ]:
(embeddings2[5]+embeddings2[6])/2

tensor([-6.2118e-02,  1.8937e-03,  2.6394e-03, -2.8366e-02,  5.7064e-01,
         5.0440e-01, -4.8092e-01, -1.4138e-01,  2.1016e-01,  5.1415e-02,
         6.2412e-01,  4.8245e-01, -1.0272e-01, -3.1518e-01,  7.9964e-02,
        -1.2104e-01,  5.0469e-01,  6.7705e-02,  1.5720e-01,  3.3734e-01,
        -6.8819e-02, -1.4083e-01, -7.3977e-01, -2.2057e-01, -3.0341e-01,
        -3.3717e-01,  3.2801e-01, -5.3635e-02, -6.3135e-01, -6.5124e-01,
         1.6604e-02,  4.0391e-01, -2.3010e-01,  4.0529e-01, -5.3306e-02,
         8.7824e-02,  9.0023e-02,  1.5853e-01,  1.9407e-01, -2.0421e-01,
        -1.3415e-02,  2.9037e-01, -5.1737e-01, -6.2468e-01,  7.4581e-02,
         4.5056e-01, -6.9139e-02, -3.8960e-01, -6.3792e-01, -4.1204e-01,
        -2.7413e-01,  1.1648e-01, -7.8523e-01,  3.2343e-01, -3.7073e-01,
         4.2116e-01, -1.2778e-01, -1.4198e-01,  1.8465e-01, -2.2483e-01,
        -1.0683e-01, -6.4681e-01,  1.3983e-01,  4.8727e-02, -6.5499e-01,
         9.8750e-02, -7.3062e-02, -3.1862e-01, -3.6

In [ ]:
lista = [1,2,3,4]

lista2 = lista[1:len(lista)-1]
lista2

[2, 3]

In [ ]:
frase = "Eu gosto muito de mamão mas o cheiro é ruim"

tokens, embeddings = tokens_embeddings(frase, tokenizer, model)

tokens

['[CLS]',
 'eu',
 'gosto',
 'muito',
 'de',
 'mam',
 '##ao',
 'mas',
 'o',
 'che',
 '##iro',
 'e',
 'ruim',
 '[SEP]']

In [ ]:
len(embeddings[2])

768

## Descobrindo um THRESHOLD pra similaridade

In [ ]:
#pra ver a similaridade
#similarity = F.cosine_similarity(embedding1, embedding2, dim=0) #pq dim =0?

'''Explicando dim=0:
embedding1 and embedding2 must be broadcastable to a common shape. <dim> refers to the dimension in this common shape.
Dimension <dim> of the output is squeezed (see torch.squeeze()), resulting in the output tensor having 1 FEWER DIMENSION.
quem tbm tem uma dimensão a menos são os operandos da similaridade de cosseno'''

corpus = '''repost from horar uniao fazer partir ir proteger rj avanco novocoronavirus troqueomedopeloscuidados
coletiva imprensar realizar tardar governador wilson witzel declarar situacao emergencia pessoal mais ficar casar isolamento domiciliar ir diferenca nesse momento poder passar coronavirus
dentro minuto fazer coletiva imprensar aqui palacio guanabara informar populacao novo medir combater coronavirus rj
viver programar novo atualizando populacao novo coronavirus
confirmar casar gravar coronavirus paciente homem cercar 60 internar hospital particular rir janeiro
olhar horaaaa ligar entrevisto viver flavio fachel situacao coronavirus apartir 6h
programacao amanhar 6h bom rir 7h entrevisto viver canal 174 sky entrar viver 90 fm 9h30 mear atualizando populacao situacao coronavirus
rjcontraocoronavirus
resultar exame dever sair atar 48 horar tratar transparencia ir atualizar informacoes qualquer ocorrencia relacionar doenca
nao ter atar nenhum confirmacao morte coronavirus rj prefeitura miguel pereira divulgar obito paciente sintoma compativeis so laboratorio central noel nutels lacen rj capaz realizar testar confirmar descartar virus
nao ter atar nenhum confirmacao morte coronavirus rj prefeitura miguel pereira divulgar obito paciente sintoma compativeis so laboratorio central noel nutels lacen rj capaz realizar testar confirmar descartar virus
obedeca quarentenar questao saude publicar somente proteger conseguir proteger populacao rjcontraocoronavirus
ficar casar apelar pai medicar gestor isolamento social voce conseguir proteger pessoa voce amo covid 19 obedeca quarentenar coronavirusnobrasil covid19
irar abastecer unidade receberao infectar coronavirus secretariar saude comprar diverso equipamento pessoal protecao epis enviar hospital parceiro
obrigar mais 10 mil profissional saude 24h voluntariaram combater coronavirus apoiar sera fundamental momento delicado saude publicar rir janeiro enfrentar'''



frase1 = "A escola de samba mocidade de padre miguel chegou perto de ganhar o carnaval"
tokens1,embeddings1 = tokens_embeddings(frase1, tokenizer, model)

frase2 = "O rio e a cidade mais bonita do brasil inteiro"
tokens2,embeddings2 = tokens_embeddings(frase2, tokenizer, model)



In [ ]:
tokens1

['[CLS]',
 'a',
 'escola',
 'de',
 'samba',
 'mo',
 '##cidade',
 'de',
 'padre',
 'mig',
 '##uel',
 'chegou',
 'ep',
 '##r',
 '##to',
 'de',
 'ganhar',
 'o',
 'carnaval',
 '[SEP]']

In [ ]:
tokens2

['[CLS]',
 'O',
 'rio',
 'e',
 'a',
 'cidade',
 'mais',
 'bonita',
 'do',
 'bras',
 '##il',
 'inteiro',
 '[SEP]']

In [ ]:
#media1 = (embeddings1[4]+embeddings1[5])/2

#media2 = (embeddings2[8]+embeddings2[9])/2

F.cosine_similarity(embeddings1[6], embeddings2[5], dim=0)

tensor(0.2554)

# Teste de performance

In [ ]:
import random


def gera_frase(corpus):
  lista_palavras = corpus.split()

  tam = random.randint(2,30)

  frase = ""

  for i in range(tam):
    indice = random.randint(0,len(lista_palavras)-1)
    palavra = lista_palavras[indice]
    frase += palavra

    if(i<(tam-1)):
      frase+=" "


  return frase



In [ ]:
frases = [gera_frase(corpus) for i in range(4989)]

In [ ]:
frases[3]

'pai rjcontraocoronavirus witzel repost divulgar rj rjcontraocoronavirus divulgar'

In [ ]:
tokenizer

BertTokenizer(name_or_path='neuralmind/bert-base-portuguese-cased', vocab_size=29794, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(29794, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
#o teste propriamente dito:

for i in range(10000):
  indice = random.randint(0,len(frases)-1)
  frase = frases[indice]
  tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

  if(i%1000 == 0):
    print(tokens)

['[CLS]', 'lac', '##en', 'ques', '##ta', '##o', 'co', '##vid', 'aman', '##har', 'ob', '##ito', 'publicar', '174', 'programa', '##ca', '##o', 'confirmar', 'programa', '##r', 'lac', '##en', 'confirma', '##ca', '##o', 'novo', '##cor', '##ona', '##vir', '##us', 'av', '##anc', '##o', 'ficar', 'divers', '##o', 'novo', '##cor', '##ona', '##vir', '##us', 'vir', '##us', 'quar', '##ente', '##nar', 'ficar', 'realizar', 'descar', '##tar', 'olhar', 'viver', '[SEP]']
['[CLS]', 'compat', '##ive', '##is', 'confirma', '##ca', '##o', 'nenhum', 'co', '##vid', '##19', 'gu', '##ana', '##bara', 'volunt', '##aria', '##ram', 'resultar', 'coro', '##na', '##vir', '##us', 'ter', 'situa', '##ca', '##o', 'fla', '##vio', 'nesse', 'canal', '[SEP]']
['[CLS]', 'co', '##vid', '##19', '60', 'nenhum', 'mil', 'somente', 'realizar', '9', '##h', '##30', 'social', 'prefeitura', 'prote', '##ca', '##o', 'hospital', '90', 'saud', '##e', 'medica', '##r', 'paciente', 'r', '##j', 'viver', 'exame', 's', '##ky', 'mig', '##uel', 'pre

### Resultado preliminar do teste de performance (feito SEM GPU):
 ~2m 30s pra tokenizar e embeddar 1000 frases.
 Isso dá 40 horas pra um milhão de frases...
 E 28h30m pra 700 mil, que é o tamanho da base grande

 ## Teste com GPU T4 do Google Colab:
   1:44s pra DEZ MIL tokenizações/embeddings...
   2h52min pra fazer um milhão.

   



# Processando as janelas com o BERT

In [ ]:
'''
Vou receber os arquivos com os textos de cada janelaa de tempo, SEM lematização
e COM stopwords
Então vou produzir novos arquivos para cada janela, sem stopwords e adaptando a tokenização
do BERT para transformar tokens cujos embeddings são parecidos
em tokens EXATAMENTE iguais no .txt

para fazer esse PROCESSAMENTO DA JANELA:
Primeiro vou tokenizar e embedar todas as frases da janela
Assim eu vou ter uma lista de listas de tokens e de embeddings
Vou popular listas auxiliares sem as stopwords, pra ser a versão
inicial do que vai pro arquivo final da janela.
Essas listas auxiliares também não terão [CLS] nem [SEP], nem nos
embeddings correspondentes

Para a primeira frase:
 Para cada token:
   Vou ver se tem algum token numa frase SEGUINTE que seja similar
    Isso será feito buscando embeddings na POSIÇÃO dos tokens
   SE SIM:
     Vou mudar na lista dos tokens todas as ocorrências similares
     pelo nome <token>+"_1" onde <token> é o token da primeira frase
     (obs: posso até mudar na lista de embeddings mas não vou precisar)
     Pra fazer essa mudança provavelmente vou gerar uma nova frase copiando
     tudo mas na posição do token em questão vou fazer a mudança

Para a segunda (e demais) frase(s):
  Para cada token que NÃO tiver um "_":
   Vou ver se tem algum token numa frase SEGUINTE que seja similar
    Isso será feito buscando embeddings na POSIÇÃO dos tokens
  SE SIM:
    Vou mudar na lista dos tokens todas as ocorrências similares
    pelo nome <token>+"_n" onde <token> é o token da primeira frase;
    E n é o número de tokens homônimos que já sofreram "justaposição"+1
    Ou seja, eu vou consultar <token>+"_1" num conjunto de tokens justapostos
    pra saber se está disponível. se não eu tento o 2 e assim por diante.
    Quando eu conseguir eu tenho que colocar <token>+"_n" no conjunto.
    Para a primeira frase também vou ter que fazer isso.


Ao final desse processo, terei a lista de lista de tokens
e vou inserir no .txt novo pra janela separando por espaço

'''

In [ ]:


def processa_janela_bert(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75):

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP]
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  k=0

  for line in arq: #aparentemente isso funciona

    frase = line.replace("\n","")
    #frase = line.split(' ')

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    tokens_pra_lista = []
    embeddings_pra_lista = []


    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    if(len(tokens_pra_lista)>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    k+=1



  arq.close()

  n_tokens = 0
  for i in range(len(lista_tokens)):
    n_tokens+= len(lista_tokens[i])

  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()


  #Agora tenho que popular a lista final de tokens

  tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  #print("Fazendo justaposição dos tokens parecidos")
  #print()
  for i in range(len(lista_tokens)): #pra cada frase
    #if((i%1000 == 0) or (i<10) ):
      #print("analisando a frase numero "+str(i))
      #print()


    frase = lista_tokens[i]
    for j in range(len(frase)): #pra cada palavra na frase
      token = lista_tokens[i][j] #vai ser uma string
      if(token.find("_") == -1): #se o token não sofreu justaposição
        #procurar nas frases SEGUINTES (e nas palavras SEGUINTES da mesma frase)
        #tokens de embeddings similares
        #OBS: esse "find" pode não ser o mais otimizado possíve, mas ainda assim deveria ser rápido...



        #Bolando qual vai ser o token correspondente caso eu ache similares
        #token_aux = token
        #k=1 #(esse aqui é o método antigo, pouco eficiente, mas que conta
        #while(True): #tenho que dar break dps #quantos tokens homonimos tem significados diferentes
         # token_aux = token
          #token_aux+="_"+str(k)
          #if(token_aux in tokens_justapostos): #vou ter que contiuar tentando
           # k+=1

          #else:
           # break #encontrei um nome apropriado,
                  #mas so vou botar o token_aux no conjunto se de fato encontrar similaridade
        token_aux = token+"_"+str(i)+"_"+str(j) #vai ser tipo "vacina_3_6"

        index_frase = j+1 #indice inicial de busca na propria frase
        #if((i==0)and(j==0)):
          #print("procurando similaridades da primeira palavra da primeira frase dentro apenas da primeira frase")
          #print()

        while(index_frase < len(frase)):
          if(similaridade(lista_embeddings[i][j],lista_embeddings[i][index_frase]) > threshold):
            #Fazendo justaposição INTRA-frase! Ou seja, colocando tokens iguais
            tokens_justapostos.add(token_aux)

            lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

            lista_tokens[i][index_frase] = token_aux


            #Vou querer deixar assim mesmo?

            #lista_embeddings[i][index_frase] = lista_embeddings[i][j]

            #isso significa que no final todos os tokens parecidos
            #vão ser associados ao mesmo embedding



          index_frase +=1

        #Já procurei dentro da frase, agora vou procurar embeddings similares em
        #outras frases. Lembrando que pode acontecer de uma frase
        #ter dois ou mais embeddings similares ao embedding atual
        #OBS: já estou com o token_aux com a numeração correta
        #Mas ainda vou ter que botar no conjunto tokens_justapostos
        #pq pode ser que nao tenha dado match dentro da


        index_lista = i+1 #procurando so nas frases seguintes pq as similaridades
                           #com frases anteriores já foram detectadas quando o loop
                           #estava na frase anterior


        while(index_lista < len(lista_tokens)): #len(lista_tokens) é a qtd de frases
          index_frase = 0 #é o index do token dentro da frase atual

          #if((i==0)and(j==0)):
           # print("procurando similaridades da primeira palavra da primeira frase dentro da frase "+str(index_lista)+" ")
            #print()

          while(index_frase< len(lista_tokens[index_lista])):

            if(similaridade(lista_embeddings[i][j],lista_embeddings[index_lista][index_frase]) > threshold):
              tokens_justapostos.add(token_aux)

              lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

              lista_tokens[index_lista][index_frase] = token_aux

              #Vou querer deixar assim mesmo?

              #lista_embeddings[index_lista][index_frase] = lista_embeddings[i][j]

              #isso significa que no final todos os tokens parecidos
              #vão ser associados ao mesmo embedding

            index_frase +=1

          index_lista +=1 #vou procurar na proxima frase



  #Saí desse forzão com a lista de tokens nova completa, pronta
  #pra COLOCAR NUM ARQUIVO NOVO REFERENTE À JANELA

  print("escrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  return


## Otimizando o processamento da janela

In [ ]:
def similarity_all_pairs(tensores):
  '''Tentativa de otimizar o cálculo da similaridade entre todos os pares de embeddings'''

  # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = tensores.norm(dim=1, keepdim=True)

  # terminando de normalizar
  data_normalized = tensores / norms

  # Calculando a similaridade de cosseno (all pairs)
  similarity_matrix = torch.matmul(data_normalized, data_normalized.T)

  return similarity_matrix


def normaliza(matriz):
  '''retorna a matriz normalizada para o cálculo da similaridade
  all pairs'''

   # Normalizando linha por linha (aqui os embeddings são vetores-linha!)
  norms = matriz.norm(dim=1, keepdim=True) #pegando a norma

  # terminando de normalizar
  data_normalized = matriz / norms


  return data_normalized

In [ ]:


def processa_janela_bert_2(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75):
  '''Essa versão é uma tentativa de otimização usando
   matmul(matriz de embeddings normalizada, matriz de embeddings normalizada transposta)
   para calcular as similaridades

   ToDo!! Adaptar pra usar o matmul de acordo com o que está nos testes
   abaixo (no meio dos testes sintaticos)

   No momento essa ainda está igual a primeira versão

   '''

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP]
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  k=0

  for line in arq: #aparentemente isso funciona

    frase = line.replace("\n","")
    #frase = line.split(' ')

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    tokens_pra_lista = []
    embeddings_pra_lista = []


    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    if(len(tokens_pra_lista)>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    k+=1



  arq.close()

  n_tokens = 0
  for i in range(len(lista_tokens)):
    n_tokens+= len(lista_tokens[i])

  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()

  #calcular similaridades aqui
  #


  #Agora tenho que popular a lista final de tokens

  tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  #print("Fazendo justaposição dos tokens parecidos")
  #print()
  for i in range(len(lista_tokens)): #pra cada frase
    #if((i%1000 == 0) or (i<10) ):
      #print("analisando a frase numero "+str(i))
      #print()


    frase = lista_tokens[i]
    for j in range(len(frase)): #pra cada palavra na frase
      token = lista_tokens[i][j] #vai ser uma string
      if(token.find("_") == -1): #se o token não sofreu justaposição
        #procurar nas frases SEGUINTES (e nas palavras SEGUINTES da mesma frase)
        #tokens de embeddings similares
        #OBS: esse "find" pode não ser o mais otimizado possíve, mas ainda assim deveria ser rápido...



        #Bolando qual vai ser o token correspondente caso eu ache similares
        #token_aux = token
        #k=1 #(esse aqui é o método antigo, pouco eficiente, mas que conta
        #while(True): #tenho que dar break dps #quantos tokens homonimos tem significados diferentes
         # token_aux = token
          #token_aux+="_"+str(k)
          #if(token_aux in tokens_justapostos): #vou ter que contiuar tentando
           # k+=1

          #else:
           # break #encontrei um nome apropriado,
                  #mas so vou botar o token_aux no conjunto se de fato encontrar similaridade
        token_aux = token+"_"+str(i)+"_"+str(j) #vai ser tipo "vacina_3_6"

        index_frase = j+1 #indice inicial de busca na propria frase
        #if((i==0)and(j==0)):
          #print("procurando similaridades da primeira palavra da primeira frase dentro apenas da primeira frase")
          #print()

        while(index_frase < len(frase)):
          if(similaridade(lista_embeddings[i][j],lista_embeddings[i][index_frase]) > threshold):
            #Fazendo justaposição INTRA-frase! Ou seja, colocando tokens iguais
            tokens_justapostos.add(token_aux)

            lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

            lista_tokens[i][index_frase] = token_aux


            #Vou querer deixar assim mesmo?

            #lista_embeddings[i][index_frase] = lista_embeddings[i][j]

            #isso significa que no final todos os tokens parecidos
            #vão ser associados ao mesmo embedding



          index_frase +=1

        #Já procurei dentro da frase, agora vou procurar embeddings similares em
        #outras frases. Lembrando que pode acontecer de uma frase
        #ter dois ou mais embeddings similares ao embedding atual
        #OBS: já estou com o token_aux com a numeração correta
        #Mas ainda vou ter que botar no conjunto tokens_justapostos
        #pq pode ser que nao tenha dado match dentro da


        index_lista = i+1 #procurando so nas frases seguintes pq as similaridades
                           #com frases anteriores já foram detectadas quando o loop
                           #estava na frase anterior


        while(index_lista < len(lista_tokens)): #len(lista_tokens) é a qtd de frases
          index_frase = 0 #é o index do token dentro da frase atual

          #if((i==0)and(j==0)):
           # print("procurando similaridades da primeira palavra da primeira frase dentro da frase "+str(index_lista)+" ")
            #print()

          while(index_frase< len(lista_tokens[index_lista])):

            if(similaridade(lista_embeddings[i][j],lista_embeddings[index_lista][index_frase]) > threshold):
              tokens_justapostos.add(token_aux)

              lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

              lista_tokens[index_lista][index_frase] = token_aux

              #Vou querer deixar assim mesmo?

              #lista_embeddings[index_lista][index_frase] = lista_embeddings[i][j]

              #isso significa que no final todos os tokens parecidos
              #vão ser associados ao mesmo embedding

            index_frase +=1

          index_lista +=1 #vou procurar na proxima frase



  #Saí desse forzão com a lista de tokens nova completa, pronta
  #pra COLOCAR NUM ARQUIVO NOVO REFERENTE À JANELA

  print("escrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  return


## Otimizando a MEMÓRIA no processamento otimizado da Janela
### Fazendo por batch

In [ ]:


def processa_janela_bert_batch(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50):
  '''O batch_size se refere à quantidade de frases em cada batch;
  A matriz original (ou melhor, aquela com um embedding por linha)
  é dividida em batches respeitando esse batch size (em media, batch_size*30 embeddings)
  E o calculo das similaridades é feito a partir de cada "bloco horizontal" da matriz
  A_norm*A_norm.T, que é a matriz com todas as similaridades par a par (esta so
   existe completa no mundo das ideias pq nao temos memoria suficiente).

  Assim, a cada rodada, temos a similaridade de todos os embeddings de cada batch entre si
  e de todos esses com os dos outros batches (mas não dos outros batches entre si,
   se não seria a matriz completa).


   Mais detalhes e desenvolvimento no caderno

   De toda forma, a cada rodada juntamos os embeddings de cada batch com eventuais próximos
   que sejam semelhantes '''

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP]
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  k=0

  for line in arq: #aparentemente isso funciona

    frase = line.replace("\n","")
    #frase = line.split(' ')

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    tokens_pra_lista = []
    embeddings_pra_lista = []


    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    if(len(tokens_pra_lista)>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    k+=1



  arq.close()

  n_tokens = 0
  for i in range(len(lista_tokens)):
    n_tokens+= len(lista_tokens[i])

  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()

  #Tenho que fazer o primeiro tensorzão aqui
  #Um embedding por linha

  tamanho_frases=[] #lista com a quantidade de tokens/embeddings em cada frase, pra facilitar a indexação
  posicao_frase = [] #posição de cada frase na matriz (tensorzão) que será construida
  posicao_frase.append(0)  #a primeira frase sempre estará na posição 0
  for i in range(len(lista_embeddings)):
    tamanho_frase = len(lista_embeddings[i])
    tamanho_frases.append(tamanho_frase)

    #aqui colocamos a posição da frase SEGUINTE baseado em onde a atual termina
    posicao_frase.append(posicao_frase[i]+tamanho_frase)

  #fazendo o tensorzao pra calcular as similaridades

  tam = len(lista_embeddings[0][0])
  print("Tamanho de cada embedding: ", tam)
  matriz = torch.zeros([n_tokens, tam], dtype=torch.float64, device='cuda')
  indice_matriz = 0
  for i in range(len(lista_embeddings)):
    for j in range(len(lista_embeddings[i])):
      matriz[indice_matriz] = lista_embeddings[i][j]
      indice_matriz += 1


  matriz_norm = normaliza(matriz) #agora tenho a matriz completa, já normalizada
  #lembrando, é um tensor
  matriz_norm.to('cuda') #já está?

  tamanho_frases_torch = torch.tensor(tamanho_frases)
  tamanho_frases_torch.to('cuda')



  #Agora tenho que popular a lista final de tokens

  tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  print("separando a matrizona em matrizinhas\n")

  #dividindo os batches
  n_frases = len(lista_tokens)
  qtd_batches = int(n_frases/batch_size) #vai arredondar pra baixo
  #essa qtd_batches tbm é a quantidade de rodadas!

  matrizinhas = [] #A_0, A_1, A_2...A_(qtd_batches-1)

  qtd_tokens_batches = [] #vou usar dps

  #fim_cada_batch = [] #vou usar dps, na hora de comparar similaridades

  fim_batch_ant = 0
  for i in range(qtd_batches):
    inicio_batch_i = fim_batch_ant

    if(i == (qtd_batches-1) ): #se é o último
      fim_batch_i =  len(matriz_norm)

      #calculando qtd_tokens_batch nesse caso
      qtd_tokens_batch = fim_batch_i- inicio_batch_i
      qtd_tokens_batches.append(qtd_tokens_batch)


    else:
      qtd_tokens_batch = 0
      #calcular a qtd de tokens nesse batch
      frase_init = int(n_frases/qtd_batches) * i

      f_atual = frase_init
      for j in range(batch_size): #são batch_size frases em cada matrizinha/ cada batch
        n_tokens_frase = tamanho_frases[f_atual]
        qtd_tokens_batch += n_tokens_frase

        f_atual +=1

      fim_batch_i = inicio_batch_i + qtd_tokens_batch
      qtd_tokens_batches.append(qtd_tokens_batch)


    #pegar efetivamente a matrizinha correspondente e botar na lista

    #os indices de slice agr sao inicio_batch_i, fim_batch_i

    matrizinha = matriz_norm[inicio_batch_i: fim_batch_i]

    matrizinhas.append(matrizinha)


  #agora falta fazer os devidos produtos e colocar
  #numa nova matriz, horizontalmente
  #   (usando torch.hstack(lista_de_tensores) )
  #é aqui que fica pesado em termos de memória



  for b in range(qtd_batches):  #b de batch
    #preciso preparar a matriz de similaridades referente a
    #linha do batch atual na matrizona de similaridades que
    #está no mundo das ideias (vide rascunho)

    #vou fazer os cálculos dos produtos de matrizes menores
    #primeiro.
    #  dps eu boto numa matriz maior que vai ser a referente a essa rodada.

    #descobrindo quantos produtos (A_i * A_j transposta)nessa rodada

    n_prod = qtd_batches - b #na rodada zero, sera qtd_batches,
                             #dps qtd_batches-1, por ai vai

    #tam_embedding = 768
    qtd_tokens_ate_final = qtd_tokens_batches[b:] #do atual pra frente
    soma_tokens_batches = sum(qtd_tokens_ate_final) #pior caso:5000*30 = 150000
                                                    #


    #pior caso: 70*30 X 70*30* qtd_batches * 8 (bytes)
    # =         70*30 X 70*30 * (5000/70) * 8 =~ 2.5 GB
    #similaridades = matriz = torch.zeros([qtd_tokens_batch, soma_tokens_batches], dtype=torch.float64, device='cuda')
    #vai ser o tensor com as similaridades
    #referentes a cada batch, dispostas horizontalmente (um A*A.T do lado do outro).
    # IMPORTANTE:
    # esse tensor será sobrescrito a cada batch.
    #É assim que economizamos memória!
    #OBS: Com o hstack, nao vou usar....

    similaridades = 1 #esse 1 é dummy, similaridades vai ser a matriz
                      #com as similaridades dessa rodada organizadas horizontalmente
    lista_tensores = [] #vou colocar os produtos aqui pra depois fazer o hstack deles

    print("Multiplicando as matrizes do batch ", b)
    print()
    for p in range(n_prod): #p de produto

      #voltar aqui depois que ja tiver separado os batches
      #vamos efetuar as multiplicações aqui

      matriz1 = matrizinhas[b]
      matriz2 = matrizinhas[b+p] #inicialmente vai ser ela com ela mesma

      produto = torch.matmul(matriz1, matriz2.T)

      lista_tensores.append(produto)

    #dps de fazer todos os produtos
    similaridades = torch.hstack(lista_tensores)



    # Falta agora fazer as junções a partir
    #das similaridades!
    #Isso será feito com os tokens da primeira frase em relaçao
    #a todas as proximas frases (é isso mesmo???pensar !)



    #print("Fazendo justaposição dos tokens parecidos")
    #print()

    #tenho que setar o começo e o fim da busca em cada
    #rodada
    #A partir de qual frase eu começo?
    #E qual a primeira frase da proximma rodada?

    frase_inicial = b*batch_size
    frase_final = (b+1)*batch_size

    # calculando frase final se estiver no ultimo batch
    if(b == qtd_batches - 1):
      frase_final = len(tamanho_frases) #a qtd de frases


    for i in range(frase_inicial, frase_final): #pra cada frase
      if((i%1000 == 0) or (i< 10 )):
        print("analisando a frase numero "+str(i))
        print()


      frase = lista_tokens[i]
      for j in range(len(frase)): #pra cada palavra na frase
        token = lista_tokens[i][j] #vai ser uma string
        if(token.find("_") == -1): #se o token não sofreu justaposição
          #procurar nas frases SEGUINTES (e nas palavras SEGUINTES da mesma frase)
          #tokens de embeddings similares
          #OBS: esse "find" pode não ser o mais otimizado possível, mas ainda assim deveria ser rápido...



          #Bolando qual vai ser o token correspondente caso eu ache similares
          #token_aux = token
          #k=1 #(esse aqui é o método antigo, pouco eficiente, mas que conta
          #while(True): #tenho que dar break dps #quantos tokens homonimos tem significados diferentes
          # token_aux = token
            #token_aux+="_"+str(k)
            #if(token_aux in tokens_justapostos): #vou ter que contiuar tentando
              # k+=1

            #else:
            # break #encontrei um nome apropriado,
                    #mas so vou botar o token_aux no conjunto se de fato encontrar similaridade
          token_aux = token+"_"+str(i)+"_"+str(j) #vai ser tipo "vacina_3_6"

          index_frase = j+1 #indice inicial de busca na propria frase
          #if((i==0)and(j==0)):
            #print("procurando similaridades da primeira palavra da primeira frase dentro apenas da primeira frase")
            #print()

          while(index_frase < len(frase)):

            #preciso procurar nos indices certos aqui
            #o certo seria, a cada batch, eu calcular a lista dos indices convertidos
            #de uma vez pra otimizar tirando vantagem da gpu...


            i_matrizona = posicao_frase[i]#o "i" na matrizona com todos os tokens um embaixo do outro
            j_matrizona = i_matrizona+j #o "j" na matrizona com todos os tokens um embaixo do outro

            i_sim = torch.sum(tamanho_frases_torch[frase_inicial:i]).item()  + j #o "i" na matriz de similaridades; se refere ao token atual
            j_sim = i_sim + (index_frase-j)#o "j" na matriz de similaridades; se refere ao token que será
                    #comparado com o atual

            if(similaridades[i_sim, j_sim] > threshold):
              #Fazendo justaposição INTRA-frase! Ou seja, colocando tokens iguais
              tokens_justapostos.add(token_aux)

              lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

              lista_tokens[i][index_frase] = token_aux


              #Vou querer deixar assim mesmo?

              #lista_embeddings[i][index_frase] = lista_embeddings[i][j]

              #isso significa que no final todos os tokens parecidos
              #vão ser associados ao mesmo embedding



            index_frase +=1

          #Já procurei dentro da frase, agora vou procurar embeddings similares em
          #outras frases. Lembrando que pode acontecer de uma frase
          #ter dois ou mais embeddings similares ao embedding atual
          #OBS: já estou com o token_aux com a numeração correta
          #Mas ainda vou ter que botar no conjunto tokens_justapostos
          #pq pode ser que nao tenha dado match dentro da


          index_lista = i+1 #procurando so nas frases seguintes pq as similaridades
                           #com frases anteriores já foram detectadas quando o loop
                           #estava na frase anterior
          #index_lista é o numero da frase com a qual eu vou comparar o token [i][j]


          while(index_lista < len(lista_tokens)): #len(lista_tokens) é a qtd de frases
            index_frase = 0 #é o index do token dentro da frase atual

            #if((i==0)and(j==0)):
            # print("procurando similaridades da primeira palavra da primeira frase dentro da frase "+str(index_lista)+" ")
              #print()

            while(index_frase< len(lista_tokens[index_lista])): #enquanto o indice na frase for menor que o tamanho da frase

              i_matrizona = posicao_frase[i]#o "i" na matrizona com todos os tokens um embaixo do outro
              j_matrizona = i_matrizona+j #o "j" na matrizona com todos os tokens um embaixo do outro

              i_sim = torch.sum(tamanho_frases_torch[frase_inicial:i]).item() + j #o "i" na matriz de similaridades; se refere ao token atual
              #acho que é esse tipo de operação que está custosa!

              index_lista_sim = torch.sum(tamanho_frases_torch[frase_inicial:index_lista]).item()
              j_sim = index_lista_sim + index_frase#o "j" na matriz de similaridades; se refere ao token que será
                    #comparado com o atual

              if(similaridades[i_sim][j_sim] > threshold):
                tokens_justapostos.add(token_aux) #acho que eu nao preciso mais disso
                #melhor tirar pq tbm pode ser custosa(?)

                lista_tokens[i][j] = token_aux #essa atribuição pode ser redundante se
                                           #for a segunda (ou "mais") similaridade com o mesmo token

                lista_tokens[index_lista][index_frase] = token_aux

                #Vou querer deixar assim mesmo?

                #lista_embeddings[index_lista][index_frase] = lista_embeddings[i][j]

                #isso significa que no final todos os tokens parecidos
                #vão ser associados ao mesmo embedding

              index_frase +=1

            index_lista +=1 #vou procurar na proxima frase
                            #e sempre vou ate a ultima!



  #Saí desse forzão com a lista de tokens nova completa, pronta
  #pra COLOCAR NUM ARQUIVO NOVO REFERENTE À JANELA

  print("escrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  return

## Testes de performance da versão batched

In [ ]:
frases = [gera_frase(corpus) for i in range(4989)]


In [ ]:
arq = open("TesteBatch.txt", "w+")



for frase in frases:
  arq.write(frase+"\n")

#tirar o ultimo '\n'?


arq.close()

In [ ]:
origem = "TesteBatch.txt"
destino = "saidaTestBatch.txt"
stopwords_file = "stopwords_pt.txt"



processa_janela_bert_batch(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50)

lendo arquivo TesteBatch.txt

lista de tokens e embeddings pronta. Restaram 142879 tokens e 4989 frases

Tamanho de cada embedding:  768
separando a matrizona em matrizinhas

Multiplicando as matrizes do batch  0

analisando a frase numero 0

analisando a frase numero 1

analisando a frase numero 2

analisando a frase numero 3

analisando a frase numero 4

analisando a frase numero 5

analisando a frase numero 6

analisando a frase numero 7

analisando a frase numero 8

analisando a frase numero 9



KeyboardInterrupt: 

## Testes com uma janela manualmente pré-processada

In [ ]:
origem = 'janela_simulada_bert.txt'
destino = 'resultTestBert.txt'
stopwords_file = 'stopwords_pt.txt'

#processa_janela_bert(origem, destino, model, tokenizer, stopwords_file)

processa_janela_bert_batch(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=70)

lendo arquivo janela_simulada_bert.txt

lista de tokens e embeddings pronta. Restaram 750 tokens e 50 frases

escrevendo a versão justaposta no arquivo resultTestBert.txt

Janela processada pronta no arquivo resultTestBert.txt


In [ ]:
origem = 'janela_simulada_bert.txt'
destino = 'resultTestBert.txt'
stopwords_file = 'stopwords_pt.txt'

#processa_janela_bert(origem, destino, model, tokenizer, stopwords_file)

processa_janela_bert_batch(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=70)

In [ ]:
frase = "auxiliar emergencial covid 19 coronavirus psolpousoalegre esquerdapousoalegre lutapousoalegre"

tokens, embeddings = tokens_embeddings(frase, tokenizer, model)

len(tokens)

28

## Otimizando MAIS ainda o processamento da janela

In [ ]:

def processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50):
  #Implementando o método novo (matriz binária * [1 2 3 4 5 6 7  ...]) eliminando zeros !!
  # Essa função faz igual as outras até a separação dos batches e multiplicação pra obter a matriz de similaridades
  # Então faz a soma esperta (1-threshold ?) em todos os elementos
  # Converte pra int pra obter a matriz binária onde 1 significa similaridade
  # multiplica linha a linha por [1 2 3 4 5 6 7 8 9 10 11 ...]
  # remove os zeros linha a linha
  # Faz a justaposição a partir das listas de similaridade obtidas

  #Para entender o batch size e demais particularidades da divisão em batches
  #ver a função processa_janela_bert_batched

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP] #lista de frases na vdd
                    #cada frase é uma lista com os tokens dessa frase
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  #DEBUG
  k=0
  linha_count = -1

  n_tokens = 0 #número total de tokens

  for line in arq: #aparentemente isso funciona
    linha_count+=1

    frase = line.replace("\n","")
    #frase = line.split(' ')

    #DEBUG
    #if(linha_count<10):
      #print("Linha "+ str(linha_count))

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    #DEBUG
    #if(linha_count < 10):
      #print("\nFrase atual ", tokens)



    tokens_pra_lista = []
    embeddings_pra_lista = []



    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    tam_lista = len(tokens_pra_lista)
    n_tokens += tam_lista

    if(tam_lista>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    k+=1



  arq.close()



  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()

  #Tenho que fazer o primeiro tensorzão aqui
  #Um embedding por linha

  #antes vou fazer a listona_tokens, onde coloco um token apos o outro,
  # pra so no final transformar de novo na lista_tokens pra botar no arquivo
  listona_tokens = []
  for frase in lista_tokens:
    listona_tokens.extend(frase)

  #debug
  #print("\n\n")
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

  lista_tokens = [] #economizando memoria


  tamanho_frases=[] #lista com a quantidade de tokens/embeddings em cada frase, pra facilitar a indexação
  posicao_frase = [] #posição do 1º token de cada frase na matriz (tensorzão) que será construida
  posicao_frase.append(0)  #a primeira frase sempre estará na posição 0
  for i in range(len(lista_embeddings)):
    tamanho_frase = len(lista_embeddings[i])
    tamanho_frases.append(tamanho_frase)

    #aqui colocamos a posição da frase SEGUINTE baseado em onde a atual termina
    posicao_frase.append(posicao_frase[i]+tamanho_frase)

  #posicao_frase vai ficar com uma entrada a mais no final
  posicao_frase = posicao_frase[:len(posicao_frase) - 1]

  #fazendo o tensorzao pra calcular as similaridades

  tam = len(lista_embeddings[0][0])
  print("Tamanho de cada embedding: ", tam)
  matriz = torch.zeros([n_tokens, tam], dtype=torch.float32, device='cuda')
  indice_matriz = 0
  for i in range(len(lista_embeddings)):
    for j in range(len(lista_embeddings[i])):
      matriz[indice_matriz] = lista_embeddings[i][j]
      indice_matriz += 1


  matriz_norm = normaliza(matriz) #agora tenho a matriz completa, já normalizada
  #lembrando, é um tensor
  matriz_norm.to('cuda') #já está?

  #tamanho_frases_torch = torch.tensor(tamanho_frases)
  #tamanho_frases_torch.to('cuda')



  #Agora tenho que popular a lista final de tokens
  #descomentar?
  # tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  #DEBUG
  print("separando a matrizona em matrizinhas\n")

  #dividindo os batches
  n_frases = len(lista_embeddings)

  print("\n\n Total de "+str(n_frases)+" frases(documentos)")
  qtd_batches = int(n_frases/batch_size) #vai arredondar pra baixo
  #essa qtd_batches tbm é a quantidade de rodadas!

  matrizinhas = [] #A_0, A_1, A_2...A_(qtd_batches-1)
  #por enquanto isso é uma lista de tensores... tensores esses que estão em GPU
  #Eu deveria fazer um tensor tri-dimensional?
  #Obs: não parece ser aqui o gargalo

  qtd_tokens_batches = [] #vou usar dps

  #fim_cada_batch = [] #vou usar dps, na hora de comparar similaridades

  fim_batch_ant = 0  #referente aos tokens
  for i in range(qtd_batches):
    inicio_batch_i = fim_batch_ant

    if(i == (qtd_batches-1) ): #se é o último
      fim_batch_i =  len(matriz_norm)

      #calculando qtd_tokens_batch nesse caso
      qtd_tokens_batch = fim_batch_i- inicio_batch_i
      qtd_tokens_batches.append(qtd_tokens_batch)


    else:
      qtd_tokens_batch = 0
      #calcular a qtd de tokens nesse batch
      frase_init = int(n_frases/qtd_batches) * i #primeira frase desse batch

      f_atual = frase_init #isso reflete a posição da frase em tamanho_frases

      for j in range(batch_size): #são batch_size frases em cada matrizinha/ cada batch
        n_tokens_frase = tamanho_frases[f_atual]
        qtd_tokens_batch += n_tokens_frase

        f_atual +=1

      fim_batch_i = inicio_batch_i + qtd_tokens_batch
      qtd_tokens_batches.append(qtd_tokens_batch)

      fim_batch_ant = fim_batch_i #pro próximo loop


    #pegar efetivamente a matrizinha correspondente e botar na lista

    #os indices de slice agr sao inicio_batch_i, fim_batch_i

    matrizinha = matriz_norm[inicio_batch_i: fim_batch_i]

    matrizinhas.append(matrizinha)

  #Obs: agr tenho em memória 2x qtd de tokens x 768 (o tamanho do tensor)
  # x 4 bytes, pe é float32

  #agora falta, em cada batch, fazer os devidos produtos e colocar
  #numa nova matriz, horizontalmente
  #   (usando torch.hstack(lista_de_tensores) )
  #é aqui que fica pesado em termos de memória


  indice_origem = 0 #indice do token "origem" na listona_tokens
                    #Vai aumentar de 1 em 1 a cada vez que olharmos
                    #pra uma lista de similaridades dentro da lista de
                    #de listas de similaridades

  conjuntos = [] #componente conexo ao qual cada token pertence
                        #considerando o grafo formado pela matriz binária
                        #de similaridades


  for i in range(n_tokens): #começa do 0, e tá tudo bem
    conjuntos.append(i)
  #originalmente, cada token pertence ao seu
  #proprio componente conexo

  conjuntos_tokens = torch.tensor(conjuntos, device='cuda')

  conjuntos = [] #jogando fora a lista, ficando so com o tensor
  posicoes_justapostas = set()

  print("Entrando no forzao")


  for b in range(qtd_batches):  #b de batch
    #preciso preparar a matriz de similaridades referente a
    #linha do batch atual na matrizona de similaridades que
    #está no mundo das ideias (vide rascunho)

    #vou fazer os cálculos dos produtos de matrizes menores
    #primeiro.
    #  dps eu boto numa matriz maior que vai ser a referente a essa rodada.

    #descobrindo quantos produtos (A_i * A_j transposta)nessa rodada

    #DEBUG
    if(b==0):
      print('\n\n primeiro batch')
    elif(b==1):
      print('\n\nsegundo batch')
    else:
      print('\n\nbatch '+str(b))


    n_prod = qtd_batches - b #na rodada zero, sera qtd_batches,
                             #dps qtd_batches-1, por ai vai

    #tam_embedding = 768
    qtd_tokens_ate_final = qtd_tokens_batches[b:] #do atual pra frente
    #qtd_tokens_ate_final é uma lista, guarda a qtd de tokens em cada batch até o final
    soma_tokens_batches = sum(qtd_tokens_ate_final) #pior caso: 2M e pouco...



    #pior caso: 70*30 X 70*30* qtd_batches * 8 (bytes)
    # =         70*30 X 70*30 * (5000/70) * 8 =~ 2.5 GB
    #similaridades = matriz = torch.zeros([qtd_tokens_batch, soma_tokens_batches], dtype=torch.float64, device='cuda')
    #vai ser o tensor com as similaridades
    #referentes a cada batch, dispostas horizontalmente (um A*A.T do lado do outro).
    # IMPORTANTE:
    # esse tensor será sobrescrito a cada batch.
    #É assim que economizamos memória!
    #OBS: Com o hstack, nao vou usar....

    similaridades = 1 #esse 1 é dummy, similaridades vai ser a matriz
                      #com as similaridades dessa rodada organizadas horizontalmente
    lista_tensores = [] #vou colocar os produtos aqui pra depois fazer o hstack deles

    #DEBUG
    #print("Multiplicando as matrizes do batch ", b)
    #print()
    for p in range(n_prod): #p de produto

      #vamos efetuar as multiplicações aqui

      matriz1 = matrizinhas[b] #fixa pra esse batch!!
      matriz2 = matrizinhas[b+p] #inicialmente vai ser ela com ela mesma

      produto = torch.matmul(matriz1, matriz2.T)

      lista_tensores.append(produto)

    #dps de fazer todos os produtos
    similaridades = torch.hstack(lista_tensores)

    #DEBUG
    #if(b<2):
      #sample = similaridades[:2]
      #print("\nsimilaridades (original): ")
      #print(sample)
      #sample = []

    lista_tensores = [] #pra economizar memoria

    #############################################################
    # Falta agora fazer as junções a partir
    #das similaridades!


    #MAS ANTES: Conversão em matriz binária, e multiplicar por [1 2 3 ...]
    # e tirar os zeros

    n_linhas = qtd_tokens_batches[b] #qtd de tokens nesse batch
    n_col = soma_tokens_batches #qtd de tokens até o final da matrizona teórica

    #matriz_p_soma = torch.ones(n_linhas, n_col, device = 'cuda') * (1-threshold)
    #ocupa espaço!!

    vetor_p_soma = torch.ones(1, n_col, device = 'cuda') * (1-threshold)


    #print(vetor_p_soma)

    similaridades = similaridades + vetor_p_soma
    #print(similaridades)

    similaridades = similaridades.int()

    #print(T)


    #FAZENDO A SEQUENCIA

    #definir n_col!!! tem que ser igual o numero de tokens!
    sequencia = torch.zeros(1,n_col, device= 'cuda')


    #num = 1
    #for i in range(len(sequencia[0])):
      #sequencia[0][i] = num
      #num +=1
    #esse método desconsidera a separação em batches!

    primeiro_token_batch_atual = 0

    if(b==0): #se é o primeiro batch
      num = 1
      for i in range(n_col):
        sequencia[0][i] = num
        num +=1
      #depois vou tirar um de cada posição nas listas de similaridades

    else: #nao precisamos mais nos preocupar com o token 0
      qtd_tokens_ate_agr = qtd_tokens_batches[:b] #vai retornar uma lista
      primeiro_token_batch_atual = sum(qtd_tokens_ate_agr)
      #posição na listona_tokens do primeiro token do batch atual

      num = primeiro_token_batch_atual
      for i in range(n_col):
        sequencia[0][i] = num
        num +=1


    #DEBUG
    #if(b<2):
      #print("sequencia")
      #print(sequencia)

    similaridades = similaridades*sequencia

    #print(similaridades)
    if(b!=0): #caso normal
      listas =[l [l !=0] for l in similaridades]



    else: #aqui b == 0 e portanto precisamos nos preocupar com o token 0
      listas =[(l [l !=0])-1 for l in similaridades]
      #obs: aqui estamos quase dobrando a memória!
      #obs2: as listas tem tamanhos diferentes
      #obs3: com o -1 no caso do primeiro batch os índices ficam corretos
      #      e capturam corretamente as similaridades pq o -1 foi feito
      #      dps de tirar os zeros!




    similaridades = [] #pra desocupar memória

    #DEBUG
    #if(b<2):
      #print("\n\n primeiras listas de similaridades:")
      #print(listas[0][:30])
      #print(listas[1][:30])


    #"pré-processamento" feito! agora tem que, a partir das
    #listas, redesignar os tokens

    frase_inicial = b*batch_size
    frase_origem = frase_inicial #frase do token "origem": estou olhando pra
                                 #lista de similaridades correspondente a esse token

    #frase_final = (b+1)*batch_size

    #A palavra atual vai ser um lista_tokens[i][j] da vida
    #Temos tamanho_frases, [] uma lista com a qtd de tokens/embeddings em cada
    #  frase;
    #  e posicao_frase, uma lista com a posição de cada frase na matriz
    # (tensorzão) construida com os embeddings um embaixo do outro
    #De posse disso e das infos de batch, e com uma lista pra cada TOKEN do tipo
    #  [1, 45, 789, 2335, 6601, 7009 ...]
    # temos que localizar as palavras origem e destino na listona_tokens pra fazer
    # as alterações necessárias
    #Com a lista_tokens pronta dps do ultimo batch, é só repetir o que é
    # feito nas outras versões dessa função!
    #------OBS:------ pra facilitar, construí uma listona de tokens,
    # "jogando fora" a informação da frase
    # E depois vou colocar os tokens dessa listona na lista_tokens original
    #usando o tamanho_frases
    #OBS: joguei isso pra antes do for, pra fazer uma vez só!
    #listona_tokens = []
    #for frase in lista_tokens:
    #  for token in frase:
    #    listona_tokens.append(token)

    #debug
    #print("\n\n")
    #print(lista_tokens[:2])
    #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

    #DEBUG
    #print("\nVerificando similaridades e redesignando tokens...")

    #0-INDEXADO!!
    #posicoes_justapostas = set() #posições na listona_tokens
                                 #dos tokens que sofreram justaposição

    #indice_origem = 0 #indice do token origem na listona_tokens
    #DEBUG                  #cuidado com o batch! resolvi acima? conferir
    #cont_debug = -1 #pra quando eu acessar pela primeira vez ser 0

    for lista_sim in listas:
      #DEBUG
      #cont_debug += 1

      if(len(lista_sim) < 2): #só é similar a ele próprio
        indice_origem +=1
        continue


      #cada lista representa um token do batch
      #aqui eu posso se quiser fazer um pequeno calculo de qual a frase
      #do token pra que a alteração do token carregue a informação da
      #frase & do token

      #a principio vai ser so o indice da listona_tokens msm

      token_origem = listona_tokens[indice_origem]

      #novo_nome_origem = token_origem #como se ja tivesse sofrido justaposição

      #if(indice_origem not in posicoes_justapostas):
        #novo_nome_origem = token_origem+"_"+str(indice_origem) #to sobrescrevendo
                                                               #a var nome_origem


      #alterou = 0
      #começar a olhar só a partir de indice_origem na lista_sim?
      conjuntos_atuais = torch.tensor([0 for i in range(len(lista_sim)) ], device='cuda')
      #o conjunto (ou componente) de cada token na lista de similaridades atual
      idx_lista_sim = 0
      for num in lista_sim: #indice do token similar ao "token origem"

        inteiro = int(num)

        conjuntos_atuais[idx_lista_sim] = conjuntos_tokens[inteiro]
        idx_lista_sim+=1

      menor = torch.min(conjuntos_atuais, dim=0)
      #vai ter "values" e "indices"
      menor_valor = int(menor.values)

      #DEBUG
      #if((b<2) and (cont_debug<2)):
        #print("\nmenor componente conexo na lista: ", menor_valor)

      indice_menor = int(menor.values)
      #indice_menor =int(menor.indices) +primeiro_token_batch_atual #errado?
      #indice na listona_tokens do token que esta associado ao componente conexo
      #de menor numero
      #OBS: O token de menor indice associado ao componente conexo de menor
      # numero (seja x esse numero) sempre estará na posição x na listona_tokens
      #por construção: todo mundo começa associado ao componente de seu próprio indice
      #e todo mundo que vai pro mesmo componente, vai pro componente de menor número

      if(indice_menor not in posicoes_justapostas):
        nome_final = listona_tokens[indice_menor]+"_"+str(menor_valor)
        posicoes_justapostas.add(indice_menor)

      else: #pra nao ficar fazendo "palavra_indice_indice_indice_indice..."
            #a cada justaposição
        nome_final = listona_tokens[indice_menor]
            #ja tem o "_indice" no final

      #DEBUG
      #if((b<2) and (cont_debug<2)):
        #print("\n Nome final dos tokens similares: ", nome_final)
      #É IMPORTANTÍSSIMO que isso aqui esteja certo pra que dê certo globalmente!!
      #Assim, todos ficam com o numero do componente conexo correspondente.
      #Que por sua vez tem esse numero, originalmente, por causa do primeiro token associado
      #a ele. Efetivamente, todos os termos semelhantes entre si ficam com o
      #indice na listona_tokens do primeiro termo que é semelhante a todos eles
      #("semelhante" aqui é quem está no mesmo componente conexo considerando
      #o grafão teórico das similaridades completas)


      for num in lista_sim:

        inteiro = int(num)

        #passando geral pro menor conjunto (o conjunto de menor numero)
        conjuntos_tokens[inteiro] = menor_valor
        #renomeando
        listona_tokens[inteiro] = nome_final
        #posicoes_justapostas.add(inteiro) #nao precisa...
        #note que, por causa da forma que a sequencia foi montada,
        #os índices já estão certos,
        #independente do batch


        #indice_destino = num


        #aqui, se o token destino já tiver sido justaposto,
        #O token_origem é que tem que mudar de nome ?????
        #listona_tokens[indice_destino] = novo_nome_origem
        #posicoes_justapostas.add(indice_destino)
        #alterou = 1

        #^^^ todo mundo que estiver na lista vai ter seu nome alterado
        #    pro nome do token original + "_posicaoOriginal"


      #saiu do for
      #if(alterou != 0): #ou seja, se rolou alguma justaposição
        #listona_tokens[indice_origem] = novo_nome_origem
        #posicoes_justapostas.add(indice_origem)



      indice_origem += 1


    #print("\n\nlistona_tokens pronta")
    #print("reconstruindo a lista_tokens")




    #ir pro próximo batch!
    #dps só falta reconstruir a janela assim como nos outros

  #Saí do forzão
  #reconstruindo a lista_tokens

  #####################01/04:verifiquei CORRETUDE até aqui, falta limpar!#######
  #DESCOMENTAR
  print("\n\n Saímos do forzão! similaridades já avaliadas\n Reconstruindo lista_tokens...")

  #DEBUG
  #time.sleep(10)

  listas = [] #lista de listas de similaridades
              #liberando a memória pois não vamos mais usar

  idx_listona = 0
  for tamanho in tamanho_frases:
    idx_frase = 0
    tokens_pra_lista = []
    while(idx_frase < tamanho):
      tokens_pra_lista.append(listona_tokens[idx_listona])
      idx_listona += 1
      idx_frase +=1

    lista_tokens.append(tokens_pra_lista) #uma frase


  #debug
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda


  print("\nescrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w+') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  return




def processa_varias_janelas(arquivos_sem_ext, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50):
  '''Processa varias janelas (os nomes dos arquivos origem estão na lista "arquivos_sem_ext", só
  com o nome, sem a extensão .txt) com o processa_janela_bert_fast; os arquivos resultantes são salvos em
  "nome_arquivo_Processada.txt"  '''

  for nome_arq in arquivos_sem_ext:
    origem = nome_arq+".txt"
    destino = nome_arq+"_Processada.txt"

    processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold, batch_size)
    torch.cuda.empty_cache()


## Testando a versão mais otimizada

In [ ]:
#lembrar de colocar arquivo de stopwords!

In [ ]:
frases = [gera_frase(corpus) for i in range(2250)]

In [ ]:
nome_arq = "TesteBatchFast.txt"

arq = open(nome_arq, "w+")



for frase in frases:
  arq.write(frase+"\n")

#tirar o ultimo '\n'?


arq.close()

In [ ]:
nome_arq = "TesteBatchFast.txt"
destino = "DestinoTesteFast.txt"
origem = nome_arq
stopwords_file = "stopwords_pt.txt"

processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=150)
torch.cuda.empty_cache()

lendo arquivo TesteBatchFast.txt

lista de tokens e embeddings pronta. Restaram 64998 tokens e 2250 frases

Tamanho de cada embedding:  768
separando a matrizona em matrizinhas



 Total de 2250 frases(documentos)
Entrando no forzao


 primeiro batch
Multiplicando as matrizes do batch  0


similaridades (original): 
tensor([[1.0000, 0.3881, 0.4373,  ..., 0.1101, 0.1614, 0.2333],
        [0.3881, 1.0000, 0.5704,  ..., 0.2675, 0.1343, 0.2778]],
       device='cuda:0')
sequencia
tensor([[1.0000e+00, 2.0000e+00, 3.0000e+00,  ..., 6.4996e+04, 6.4997e+04,
         6.4998e+04]], device='cuda:0')


 primeiras listas de similaridades:
tensor([    0.,   223.,   815.,   897.,  3049.,  3448.,  6802., 10036., 10838.,
        10988., 11204., 13373., 14858., 15129., 16205., 17725., 20488., 23408.,
        24799., 28557., 30448., 31371., 33087., 33762., 35814., 36963., 37326.,
        37697., 38292., 38354.], device='cuda:0')
tensor([1.0000e+00, 8.1600e+02, 8.9800e+02, 1.0420e+03, 2.5170e+03, 2.6460e+

In [ ]:
#Testando com uma janela real!! quase 10.000 tokens

nome_arq = "T31.txt"
destino = "T31_Processada.txt"
origem = nome_arq
stopwords_file = "stopwords_pt.txt"

processa_janela_bert_fast(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=50)
torch.cuda.empty_cache()

FileNotFoundError: [Errno 2] No such file or directory: 'T31.txt'

In [ ]:
#Processando varias janelas
#Lembrar de uploadar os arquivos correspondentes e o de stopwords

arquivos_sem_ext = ["T120", "T121", "T122"]
stopwords_file = "stopwords_pt.txt"

processa_varias_janelas(arquivos_sem_ext, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=80)

lendo arquivo T120.txt

lista de tokens e embeddings pronta. Restaram 155126 tokens e 9999 frases

Tamanho de cada embedding:  768
separando a matrizona em matrizinhas



 Total de 9999 frases(documentos)
Entrando no forzao


 primeiro batch


segundo batch


batch 2


batch 3


batch 4


batch 5


batch 6


batch 7


batch 8


batch 9


batch 10


batch 11


batch 12


batch 13


batch 14


batch 15


batch 16


batch 17


batch 18


batch 19


batch 20


batch 21


batch 22


batch 23


batch 24


batch 25


batch 26


batch 27


batch 28


batch 29


batch 30


batch 31


batch 32


batch 33


batch 34


batch 35


batch 36


batch 37


batch 38


batch 39


batch 40


batch 41


batch 42


batch 43


batch 44


batch 45


batch 46


batch 47


batch 48


batch 49


batch 50


batch 51


batch 52


batch 53


batch 54


batch 55


batch 56


batch 57


batch 58


batch 59


batch 60


batch 61


batch 62


batch 63


batch 64


batch 65


batch 66


batch 67


batch 68


batch 69




### Limpando a versão mais otimizada

In [ ]:
#To-Do: colar a processa_janela_bert_fast aqui e limpar!!!

## Otimização EXTRA no processa_janela

Fazendo o batch nas duas dimensões

In [ ]:

#OBS: a parte do "componentes_bacthes" aqui provavelmente não tá fazendo nada de bom.
#     Tirar?

def processa_janela_bert_extra(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=1500):
  #aqui o batch size na vdd é a quantidade de TOKENS em cada batch
  #vou fazer a similaridade dos primeiros batch_size tokens consigo,
  #dps destes com os proximos batch_size tokens (portanto andando pra direita
  #na matriz grandona conceitual de similaridades)
  #e dps dos "segundos" batch_size tokens consigo e com os outros

  set_stopwords = get_stopwords(stopwords_file)


  arq = open(origem, "r")

  #lista_tokens_originais = []
  #lista_embeddings_originais = []

  #lista_tokens_final = []
  #lista_embeddings_final = []

  lista_tokens = [] #sem stopwords nem [CLS] ou [SEP] #lista de frases na vdd
                    #cada frase é uma lista com os tokens dessa frase
  lista_embeddings = []


  print("lendo arquivo "+origem)
  print()#\n

  #DEBUG
  #k=0
  #linha_count = -1

  n_tokens = 0 #número total de tokens

  for line in arq: #aparentemente isso funciona
    #linha_count+=1

    frase = line.replace("\n","")
    #frase = line.split(' ')

    #DEBUG
    #if(linha_count<10):
      #print("Linha "+ str(linha_count))

    tokens,embeddings = tokens_embeddings(frase, tokenizer, model)

    #DEBUG
    #if(linha_count < 10):
      #print("\nFrase atual ", tokens)



    tokens_pra_lista = []
    embeddings_pra_lista = []



    for i in range(len(tokens)):
      if( not( (tokens[i] == '[CLS]') or (tokens[i] == '[SEP]') or (tokens[i] in set_stopwords)) ):
        #ou seja, se é um token "válido"
        tokens_pra_lista.append(tokens[i])
        embeddings_pra_lista.append(embeddings[i]) #o embedding correspondente

    tam_lista = len(tokens_pra_lista)
    n_tokens += tam_lista

    if(tam_lista>0): #tem que ter sobrado algum token
      lista_tokens.append(tokens_pra_lista)
      lista_embeddings.append(embeddings_pra_lista)

      #if(k==0):
       # print("frase demo: ")
        #print(tokens_pra_lista)

    #k+=1



  arq.close()



  print("lista de tokens e embeddings pronta. Restaram "+str(n_tokens)+" tokens e "+str(len(lista_tokens))+" frases")
  print()

  #Tenho que fazer o primeiro tensorzão aqui
  #Um embedding por linha

  #antes vou fazer a listona_tokens, onde coloco um token apos o outro,
  # pra so no final transformar de novo na lista_tokens pra botar no arquivo
  listona_tokens = []
  for frase in lista_tokens:
    listona_tokens.extend(frase)

  #debug
  #print("\n\n")
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

  lista_tokens = [] #economizando memoria


  tamanho_frases=[] #lista com a quantidade de tokens/embeddings em cada frase, pra facilitar a indexação
  posicao_frase = [] #posição do 1º token de cada frase na matriz (tensorzão) que será construida
  posicao_frase.append(0)  #a primeira frase sempre estará na posição 0
  for i in range(len(lista_embeddings)):
    tamanho_frase = len(lista_embeddings[i])
    tamanho_frases.append(tamanho_frase)

    #aqui colocamos a posição da frase SEGUINTE baseado em onde a atual termina
    posicao_frase.append(posicao_frase[i]+tamanho_frase)

  #posicao_frase vai ficar com uma entrada a mais no final
  posicao_frase = posicao_frase[:len(posicao_frase) - 1]

  #fazendo o tensorzao pra calcular as similaridades

  tam = len(lista_embeddings[0][0])
  #print("Tamanho de cada embedding: ", tam)
  matriz = torch.zeros([n_tokens, tam], dtype=torch.float32, device='cuda')
  indice_matriz = 0
  for i in range(len(lista_embeddings)):
    for j in range(len(lista_embeddings[i])):
      matriz[indice_matriz] = lista_embeddings[i][j]
      indice_matriz += 1


  matriz_norm = normaliza(matriz) #agora tenho a matriz completa, já normalizada
  #lembrando, é um tensor
  matriz_norm.to('cuda') #já está?

  #tamanho_frases_torch = torch.tensor(tamanho_frases)
  #tamanho_frases_torch.to('cuda')



  #Agora tenho que popular a lista final de tokens
  #descomentar?
  # tokens_justapostos = set() #os tokens do tipo <token>_n, que surgiram
                             #apos justaposição serão salvos aqui
                             #isso serve pq as vezes tokens homonimos
                             #NÃO serão justapostos

  #...qual é a complexidade disso?...

  #DEBUG
  #print("separando a matrizona em matrizinhas\n")

  #dividindo os batches
  #n_frases = len(lista_embeddings)

  #print("\n\n Total de "+str(n_frases)+" frases(documentos)")
  qtd_batches = int(n_tokens/batch_size) #vai arredondar pra baixo
  #essa qtd_batches tbm é a quantidade de rodadas!

  matrizinhas = [] #A_0, A_1, A_2...A_(qtd_batches-1)
  #por enquanto isso é uma lista de tensores... tensores esses que estão em GPU
  #Eu deveria fazer um tensor tri-dimensional?
  #Obs: não parece ser aqui o gargalo

  qtd_tokens_batches = [] #vou usar dps

  #fim_cada_batch = [] #vou usar dps, na hora de comparar similaridades

  fim_batch_ant = 0  #referente aos tokens
  for i in range(qtd_batches):
    inicio_batch_i = fim_batch_ant

    if(i == (qtd_batches-1) ): #se é o último
      fim_batch_i =  len(matriz_norm)

      #calculando qtd_tokens_batch nesse caso
      qtd_tokens_batch = fim_batch_i- inicio_batch_i
      qtd_tokens_batches.append(qtd_tokens_batch)


    else: #isso aqui vai mudar
      qtd_tokens_batch = 0
      #calcular a qtd de tokens nesse batch
      #token_init = int(n_tokens/qtd_batches) * i #primeiro token desse batch
      #Não vou precisar disso pq a conta vai bater...TEM QUE bater!!!
      #inicio_batch_i = token_init ja ta declarado



      fim_batch_i = inicio_batch_i + batch_size
      qtd_tokens_batches.append(batch_size)

      fim_batch_ant = fim_batch_i #pro próximo loop


    #pegar efetivamente a matrizinha correspondente e botar na lista

    #os indices de slice agr sao inicio_batch_i, fim_batch_i

    matrizinha = matriz_norm[inicio_batch_i: fim_batch_i]

    matrizinhas.append(matrizinha)

  #Obs: agr tenho em memória 2x qtd de tokens x 768 (o tamanho do tensor)
  # x 4 bytes, pe é float32

  #agora falta, em cada batch, fazer os devidos produtos e colocar
  #numa nova matriz, horizontalmente
  #   (usando torch.hstack(lista_de_tensores) )
  #é aqui que fica pesado em termos de memória


  indice_origem = 0 #indice do token "origem" na listona_tokens
                    #Vai aumentar de 1 em 1 a cada vez que olharmos
                    #pra uma lista de similaridades dentro da lista de
                    #de listas de similaridades

  conjuntos = [] #componente conexo ao qual cada token pertence
                        #considerando o grafo formado pela matriz binária
                        #de similaridades


  for i in range(n_tokens): #começa do 0, e tá tudo bem
    conjuntos.append(i)
  #originalmente, cada token pertence ao seu
  #proprio componente conexo

  conjuntos_tokens = torch.tensor(conjuntos, device='cuda')

  conjuntos = [] #jogando fora a lista, ficando so com o tensor

  posicoes_justapostas = set()

  print("Entrando no forzao")


  for b in range(qtd_batches):  #b de batch #esse aqui é o outer loop
    #preciso preparar a matriz de similaridades referente a
    #linha do batch atual na matrizona de similaridades que
    #está no mundo das ideias (vide rascunho)

    #vou fazer os cálculos dos produtos de matrizes menores
    #primeiro.
    #  dps eu boto numa matriz maior que vai ser a referente a essa rodada.

    #descobrindo quantos produtos (A_i * A_j transposta)nessa rodada

    #DEBUG
    #if(b==0):
    #  print('\n\n primeiro batch')
    #elif(b==1):
    #  print('\n\nsegundo batch')

    componentes_batch = torch.ones(1, batch_size, device = 'cuda') * 999999999
    #pensar nisso na vdd como vetor em pé
    #nunca chegaremos a 1 bilhão de tokens na janela então ta safe esse número


    n_prod = qtd_batches - b #na rodada zero, sera qtd_batches,
                             #dps qtd_batches-1, por ai vai

    #tam_embedding = 768
    #qtd_tokens_ate_final = qtd_tokens_batches[b:] #do atual pra frente
    #qtd_tokens_ate_final é uma lista, guarda a qtd de tokens em cada batch até o final
    #soma_tokens_batches = sum(qtd_tokens_ate_final) #pior caso: 2M e pouco...



    #pior caso: 70*30 X 70*30* qtd_batches * 8 (bytes)
    # =         70*30 X 70*30 * (5000/70) * 8 =~ 2.5 GB
    #similaridades = matriz = torch.zeros([qtd_tokens_batch, soma_tokens_batches], dtype=torch.float64, device='cuda')
    #vai ser o tensor com as similaridades
    #referentes a cada batch, dispostas horizontalmente (um A*A.T do lado do outro).
    # IMPORTANTE:
    # esse tensor será sobrescrito a cada batch.
    #É assim que economizamos memória!
    #OBS: Com o hstack, nao vou usar....

    similaridades = 1 #esse 1 é dummy, similaridades vai ser a matriz
                      #com as similaridades dessa rodada organizadas horizontalmente
    #lista_tensores = [] #vou colocar os produtos aqui pra depois fazer o hstack deles

    #DEBUG
    #print("Multiplicando as matrizes do batch ", b)
    #print()
    #esse é o inner loop
    #vou ter que dar tab em um monte de gente ali pq vai ser um produto por vez
    ############PAREI AQUI!!##########################
    for p in range(n_prod): #p de produto
      #Vai ser um produto por vez; vou calcular as similaridades
      #e determinar os componentes de cada um, bem como o novo nome dos tokens
      #OBS: por causa da divisão que estamos fazendo um token pode mudar de nome
      #algumas vezes

      #vamos efetuar as multiplicações aqui

      matriz1 = matrizinhas[b] #fixa pra esse batch!!
      matriz2 = matrizinhas[b+p] #inicialmente vai ser ela com ela mesma

      similaridades = torch.matmul(matriz1, matriz2.T)

      #lista_tensores.append(produto)

      #dps de fazer todos os produtos
      #similaridades = torch.hstack(lista_tensores)

      #DEBUG
      #if(b<2):
        #sample = similaridades[:2]
        #print("\nsimilaridades (original): ")
        #print(sample)
        #sample = []

      #lista_tensores = [] #pra economizar memoria

      #############################################################
      # Falta agora fazer as junções a partir
      #das similaridades!


      #MAS ANTES: Conversão em matriz binária, e multiplicar por [1 2 3 ...]
      # e tirar os zeros
      n_linhas,n_col = similaridades.size()
      #n_linhas = qtd_tokens_batches[b] #qtd de tokens nesse batch
      #n_col = soma_tokens_batches #qtd de tokens até o final da matrizona teórica

      #matriz_p_soma = torch.ones(n_linhas, n_col, device = 'cuda') * (1-threshold)
      #ocupa espaço!!

      vetor_p_soma = torch.ones(1, n_col, device = 'cuda') * (1-threshold)


      #print(vetor_p_soma)

      similaridades = similaridades + vetor_p_soma
      #print(similaridades)

      similaridades = similaridades.int()

      #print(T)


      #FAZENDO A SEQUENCIA

      #definimos n_col!!! tem que ser igual o numero de tokens no batch!
      sequencia = torch.zeros(1,n_col, device= 'cuda')


      #num = 1
      #for i in range(len(sequencia[0])):
        #sequencia[0][i] = num
        #num +=1
      #esse método desconsidera a separação em batches!

      #primeiro_token_batch_atual = 0
      primeiro_token_outer_loop = 0
      primeiro_token_inner_loop = 0

      if(b==0 and p==0): #se é o primeiro batch
        num = 1
        for i in range(n_col):
          sequencia[0][i] = num
          num +=1
        #depois vou tirar um de cada posição nas listas de similaridades

      else: #nao precisamos mais nos preocupar com o token 0
        #qtd_tokens_ate_agr = qtd_tokens_batches[:b] #vai retornar uma lista
        #primeiro_token_batch_atual = sum(qtd_tokens_ate_agr)
        #posição na listona_tokens do primeiro token do batch atual
        #alternativamente é simplesmente batch_size * b + batch_size * p

        primeiro_token_outer_loop = batch_size*b
        primeiro_token_inner_loop = primeiro_token_outer_loop + batch_size*p

        num = primeiro_token_inner_loop
        for i in range(n_col):
          sequencia[0][i] = num
          num +=1
          #ex:primeiro token outer loop = 200
          #   primeiro token inner loop  = 300
          #Vamos descobrir com essa sequencia se o 300, 301, 302, etc
          #são similares ao 200
          #na segunda linha dessa matrizinha de similaridades, vamos descobrir
          #se o 300, 301, 302 são similares ao 201


      #DEBUG
      #if(p<2):
        #print("sequencia")
        #print(sequencia)

      similaridades = similaridades*sequencia

      #print(similaridades)
      if((b!=0) or (p!=0)): #caso normal
        listas =[l [l !=0] for l in similaridades]



      else: #aqui b == 0 e portanto precisamos nos preocupar com o token 0
        listas =[(l [l !=0])-1 for l in similaridades]
        #obs: aqui estamos quase dobrando a memória!
        #obs2: as listas tem tamanhos diferentes
        #obs3: com o -1 no caso do primeiro batch os índices ficam corretos
        #      e capturam corretamente as similaridades pq o -1 foi feito
        #      dps de tirar os zeros!




      similaridades = [] #pra desocupar memória

      #DEBUG
      #if(b<2):
        #print("\n\n primeiras listas de similaridades:")
        #print(listas[0][:30])
        #print(listas[1][:30])


      #"pré-processamento" feito! agora tem que, a partir das
      #listas, redesignar os tokens

      frase_inicial = b*batch_size
      frase_origem = frase_inicial #frase do token "origem": estou olhando pra
                                 #lista de similaridades correspondente a esse token

      #frase_final = (b+1)*batch_size
      #pra ver o comentário que estava aqui (textão), olhar
      #



      #debug
      #print("\n\n")
      #print(lista_tokens[:2])
      #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda

      #DEBUG
      #print("\nVerificando similaridades e redesignando tokens...")

      #0-INDEXADO!!
      #posicoes_justapostas = set() #posições na listona_tokens
                                 #dos tokens que sofreram justaposição

      #indice_origem = 0 #indice do token origem na listona_tokens
      #DEBUG                  #cuidado com o batch! resolvi acima? conferir
      #cont_debug = -1 #pra quando eu acessar pela primeira vez ser 0
      offset_origem = 0
      for lista_sim in listas:
        #DEBUG
        #cont_debug += 1

        if(len(lista_sim) < 2): #só é similar a ele próprio
          indice_origem +=1
          offset_origem += 1
          continue


        #cada lista representa um token do batch
        #aqui eu posso se quiser fazer um pequeno calculo de qual a frase
        #do token pra que a alteração do token carregue a informação da
        #frase & do token

        #a principio vai ser so o indice da listona_tokens msm

        #token_origem = listona_tokens[indice_origem]
        token_origem = primeiro_token_outer_loop + offset_origem
        #"estamos olhando pra lista de quem?"

        #novo_nome_origem = token_origem #como se ja tivesse sofrido justaposição

        #if(indice_origem not in posicoes_justapostas):
          #novo_nome_origem = token_origem+"_"+str(indice_origem) #to sobrescrevendo
                                                               #a var nome_origem


        #alterou = 0
        #começar a olhar só a partir de indice_origem na lista_sim?
        conjuntos_atuais = torch.zeros((len(lista_sim)+2), device='cuda')
        #o conjunto (ou componente) de cada token na lista de similaridades atual
        #+ o conjunto do token_origem
        #+ o menor conjunto da linha inteira da matrizona teórica de similaridades,
        #  em componentes_batch[offset_origem]

        idx_lista_sim = 0
        for num in lista_sim: #indice do token similar ao "token origem"

          inteiro = int(num)

          conjuntos_atuais[idx_lista_sim] = conjuntos_tokens[inteiro]
          idx_lista_sim+=1
        #findo o for:
        conjuntos_atuais[idx_lista_sim] = conjuntos_tokens[token_origem]
        conjuntos_atuais[idx_lista_sim+1] = componentes_batch[offset_origem]



        menor = torch.min(conjuntos_atuais, dim=0)
        #vai ter "values" e "indices"
        menor_valor = int(menor.values)

        #DEBUG
        #if((b<2) and (cont_debug<2) and(p<2)):
          #print("\nmenor componente conexo na lista: ", menor_valor)

        componentes_batch[offset_origem] = menor_valor
        #é essa atualização aqui nesse chunk do inner loop que vai propagar, se
        #necessário, o menor componente da linha INTEIRA aos tokens
        #que estiverem mais a frente na linha (?????)

        indice_menor = int(menor.values)
        #indice_menor =int(menor.indices) +primeiro_token_batch_atual #errado?
        #indice na listona_tokens do token que esta associado ao componente conexo
        #de menor numero
        #OBS: O token de menor indice associado ao componente conexo de menor
        # numero (seja x esse numero) sempre estará na posição x na listona_tokens
        #por construção: todo mundo começa associado ao componente de seu próprio indice
        #e todo mundo que vai pro mesmo componente, vai pro componente de menor número

        if(indice_menor not in posicoes_justapostas):
          nome_final = listona_tokens[indice_menor]+"_"+str(menor_valor)
          posicoes_justapostas.add(indice_menor)

        else: #pra nao ficar fazendo "palavra_indice_indice_indice_indice..."
              #a cada justaposição
          nome_final = listona_tokens[indice_menor]
              #ja tem o "_indice" no final

        #DEBUG
        #if((b<2) and (cont_debug<2)):
          #print("\n Nome final dos tokens similares: ", nome_final)
        #É IMPORTANTÍSSIMO que isso aqui esteja certo pra que dê certo globalmente!!
        #Assim, todos ficam com o numero do componente conexo correspondente.
        #Que por sua vez tem esse numero, originalmente, por causa do primeiro token associado
        #a ele. Efetivamente, todos os termos semelhantes entre si ficam com o
        #indice na listona_tokens do primeiro termo que é semelhante a todos eles
        #("semelhante" aqui é quem está no mesmo componente conexo considerando
        #o grafão teórico das similaridades completas)


        for num in lista_sim:

          inteiro = int(num)

          #passando geral pro menor conjunto (o conjunto de menor numero)
          conjuntos_tokens[inteiro] = menor_valor
          #renomeando
          listona_tokens[inteiro] = nome_final
          #posicoes_justapostas.add(inteiro) #nao precisa...
          #note que, por causa da forma que a sequencia foi montada,
          #os índices já estão certos,
          #independente do batch


          #indice_destino = num


          #aqui, se o token destino já tiver sido justaposto,
          #O token_origem é que tem que mudar de nome ?????
          #listona_tokens[indice_destino] = novo_nome_origem
          #posicoes_justapostas.add(indice_destino)
          #alterou = 1

          #^^^ todo mundo que estiver na lista vai ter seu nome alterado
          #    pro nome do token original + "_posicaoOriginal"


        #saiu do for
        #if(alterou != 0): #ou seja, se rolou alguma justaposição
          #listona_tokens[indice_origem] = novo_nome_origem
          #posicoes_justapostas.add(indice_origem)



        offset_origem += 1


    #print("\n\nlistona_tokens pronta")
    #print("reconstruindo a lista_tokens")




    #ir pro próximo batch!
    #dps só falta reconstruir a janela assim como nos outros

  #Saí do forzão
  #reconstruindo a lista_tokens

  #####################01/04:verifiquei CORRETUDE até aqui, falta limpar!#######
  #DESCOMENTAR
  print("\n\n Saímos do forzão! similaridades já avaliadas\n Reconstruindo lista_tokens...")

  #DEBUG
  #time.sleep(10)

  listas = [] #lista de listas de similaridades
              #liberando a memória pois não vamos mais usar

  idx_listona = 0
  for tamanho in tamanho_frases:
    idx_frase = 0
    tokens_pra_lista = []
    while(idx_frase < tamanho):
      tokens_pra_lista.append(listona_tokens[idx_listona])
      idx_listona += 1
      idx_frase +=1

    lista_tokens.append(tokens_pra_lista) #uma frase


  #debug
  #print(lista_tokens[:2])
  #print(listona_tokens([:45])) #provavelmente a 1ª frase e metade da segunda


  print("\nescrevendo a versão justaposta no arquivo "+destino)
  print()
  arq_dest = open(destino, 'w+') #cria o arquivo para escrita

  for i in range(len(lista_tokens)):
    frase = lista_tokens[i]
    for j in range(len(frase)):
      arq_dest.write(frase[j])
      if(j<len(frase)-1):
        arq_dest.write(" ")

    if(i<len(lista_tokens)-1):
      arq_dest.write("\n")


  arq_dest.close()
  print("Janela processada pronta no arquivo "+ destino)

  lista_tokens = []
  listona_tokens = []

  return


###Testando a otimização EXTRA

In [ ]:
#tem que gerar o arquivo ali em cima
destino = "DestinoTesteExtra.txt"
origem = "TesteBatchFast.txt"
stopwords_file = "stopwords_pt.txt"

processa_janela_bert_extra(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=1500)

lendo arquivo TesteBatchFast.txt

lista de tokens e embeddings pronta. Restaram 28756 tokens e 1001 frases

Tamanho de cada embedding:  768
separando a matrizona em matrizinhas

Entrando no forzao


 primeiro batch
Multiplicando as matrizes do batch  0


similaridades (original): 
tensor([[1.0000, 0.2898, 0.2701,  ..., 0.3091, 0.0920, 0.1897],
        [0.2898, 1.0000, 0.6854,  ..., 0.2866, 0.2221, 0.0821]],
       device='cuda:0')
sequencia
tensor([[1.0000e+00, 2.0000e+00, 3.0000e+00,  ..., 1.4980e+03, 1.4990e+03,
         1.5000e+03]], device='cuda:0')


 primeiras listas de similaridades:
tensor([   0.,  714., 1077.], device='cuda:0')
tensor([  1.,  55.,  68.,  77.,  82., 125., 179., 248., 265., 270., 307., 348.,
        433., 555., 560., 584., 652., 659., 715., 730., 733., 755., 761., 780.,
        818., 895., 946., 962., 983., 991.], device='cuda:0')

Verificando similaridades e redesignando tokens...

menor componente conexo na lista:  0

 Nome final dos tokens similares:  program

In [ ]:
#Mais testes de performance
#tem que dar o shift enter na célula do corpus

frases = [gera_frase(corpus) for i in range(2250)]

In [ ]:
nome_arq = "TestePerformanceExtra.txt"

arq = open(nome_arq, "w+")



for frase in frases:
  arq.write(frase+"\n")

#tirar o ultimo '\n'?


arq.close()

In [ ]:
origem = nome_arq
destino = "DestinoTesteExtra.txt"
stopwords_file = "stopwords_pt.txt"

processa_janela_bert_extra(origem, destino, model, tokenizer, stopwords_file, threshold = 0.75, batch_size=1000)
torch.cuda.empty_cache()

lendo arquivo TestePerformanceExtra.txt

lista de tokens e embeddings pronta. Restaram 64086 tokens e 2250 frases

Entrando no forzao


 Saímos do forzão! similaridades já avaliadas
 Reconstruindo lista_tokens...

escrevendo a versão justaposta no arquivo DestinoTesteExtra.txt

Janela processada pronta no arquivo DestinoTesteExtra.txt


# Testes sintáticos (?)

In [ ]:
#ToDo: (geral)
#Ver se BERT entende ponto final(vide inicio do notebook)
#  ver http://www.apsipa.org/proceedings/2019/pdfs/180.pdf
#adaptar as funções atuais do VERSATILE pra
#nao chamar a lematização(? é só eu nao chamar)
#e pra manter as stopwords
#extrair de novo da base

#executar o VERSATILE!



In [ ]:
frases = "Outro dia falei com sua mãe. Ela me disse que você tinha ido morar fora"
tokens, embeddings = tokens_embeddings(frases, tokenizer, model)

In [ ]:
tokens

['[CLS]',
 'outro',
 'dia',
 'fale',
 '##i',
 'com',
 'sua',
 'ma',
 '##e',
 '.',
 'ela',
 'me',
 'disse',
 'que',
 'voc',
 '##e',
 'tinha',
 'ido',
 'morar',
 'fora',
 '[SEP]']

In [ ]:
frases = [gera_frase(corpus) for i in range(489)]

NameError: name 'gera_frase' is not defined

In [ ]:
lista_tokens = []
lista_embeddings = []

for frase in frases:
  tokens, embeddings = tokens_embeddings(frase, tokenizer, model)
  lista_tokens.append(tokens)
  lista_embeddings.append(embeddings)

#aparentemente os embeddings tem 768 elementos
frase_embedada = lista_embeddings[0]



In [ ]:
lista = [random.randint(1,700000) for i in range(5000)]


lista_torch = torch.tensor(lista)
lista_torch.to("cuda")

sum(lista_torch[1000:]).item()



1383152324

In [ ]:
#testando performance
#com pytorch

n_casos = 10000000

d = 0

for i in range(n_casos):
  cut = random.randint(1,len(lista_torch)-1)

  d = torch.sum(lista_torch[cut:]).item()


print(d)





634547692


In [ ]:
#testando performance
#sem pytorch

n_casos = 1000000

d = 0

for i in range(n_casos):
  cut = random.randint(1,len(lista)-1)

  d = sum(lista[cut:])


print(d)

1243442251


## Calculando similaridade par a par e acessando os resultados

In [ ]:
#Aqui eu estou fazendo um tensorzão com um embedding embaixo do outro,
# (um embedding por linha)
# ordenado por frase a partir da matriz original,
# que tem uma frase (vários embeddings) por linha
#Nisso também é preciso fazer certos cálculos (por exemplo,
#quantos tokens em cada frase), que estão aqui nesta célula
#(!) Isso será reaproveitado na função eficiente do processa_janela_bert


#lista_cpu = [tensor.cpu() for tensor in lista_embeddings]
#lista_cpu
#matriz = np.array(lista_cpu)
#tenho que DECIDIR aqui entre popular um tensorzão com os tensores
#existentes, o que vai dificultar a indexação depois pq o número de embeddings
#pra cada frase varia;
#Ou preencher a lista de embeddings com tensores nulos, o que vai dificultar o cálculo
#mas vai facilitar a indexação SE eu utilizar os tamanhos originais pra buscar
#e vai facilitar a ciração(?) do tensorzão
#Já decidi pela primeira estratégia


#contando embeddings
n_embeddings = 0
for i in range(len(lista_embeddings)):
    n_embeddings+= len(lista_embeddings[i])

#tam_igual = 1
tam = len(lista_embeddings[0][0])
#print("tamanho do primeiro tensor: ", tam)
#for i in range(len(lista_cpu)):
#  for j in range(len(lista_cpu[i])):
#    tam2 = len(lista_cpu[i][j])

#    if(not(tam2==tam)):
#      tam_igual = 0
#      break

#print(tam_igual)

tamanho_frases=[] #lista com a quantidade de tokens/embeddings em cada frase, pra facilitar a indexação
posicao_frase = [] #posição de cada frase na matriz (tensorzão) que será construida
posicao_frase.append(0)  #a primeira frase sempre estará na posição 0
for i in range(len(lista_embeddings)):
  tamanho_frase = len(lista_embeddings[i])
  tamanho_frases.append(tamanho_frase)

  #aqui colocamos a posição da frase SEGUINTE baseado em onde a atual termina
  posicao_frase.append(posicao_frase[i]+tamanho_frase)

#fazendo o tensorzao pra calcular as similaridades
matriz = torch.zeros([n_embeddings, tam], dtype=torch.float64, device='cuda')
indice_matriz = 0
for i in range(len(lista_embeddings)):
  for j in range(len(lista_embeddings[i])):
    matriz[indice_matriz] = lista_embeddings[i][j]
    indice_matriz += 1


matriz

tensor([[ 0.1265, -0.6420,  0.7438,  ..., -0.2870,  0.1130, -0.4518],
        [ 0.6302, -0.6198,  0.5248,  ...,  0.0291, -0.0422, -0.6594],
        [ 0.4947,  0.0269,  0.6759,  ...,  0.3091,  0.1976, -0.1915],
        ...,
        [ 0.0754, -0.0391,  0.0851,  ..., -0.4702,  0.1336, -0.2869],
        [ 0.0537, -0.0643,  1.0996,  ...,  0.2881,  0.5409, -0.4310],
        [-0.0762, -0.1146,  0.6665,  ..., -0.3545,  0.2711, -0.8931]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
sim = similarity_all_pairs(matriz)

In [ ]:
#pra resolver isso eu posso fazer o calculo do all pairs varias vezes, uma pra cada frase por exemplo
#Ou melhor, pra cada frase i da janela, pra cada frase k que eu esteja olhando,
#eu posso gerar uma matriz que tem todos os tensores da primeira frase e da segunda
# e então olhar as similaridades so dos x primeiros indices da matriz de resultado,
# onde x é q quantidade de embeddings da frase i.

#...isso significa que eu faria O(m^2) matmuls pequenos, onde m é o número de tweets
#Não confundir com O(n^2) operações, que é o caso do primeiro algoritmo, ONDE n
# É O NÚMERO DE TOKENS NA JANELA.
#

In [ ]:
#pegando a similaridade da palavra i,j com a palavra k,l

i = 1
j = 2 #palavra 2 da frase 1 #deveria ser o indice 29?

k = 2
l = 3 #palavra 2 da frase 9  # deveria ser o indice 27+42+3 = 72?


indice_i_j= posicao_frase[i] + j
print(indice_i_j)

indice_k_l = posicao_frase[k] + l
print(indice_k_l)

sim[indice_i_j][indice_k_l] #é um tensor, tem que pegar item() pro numero

29
72


tensor(0.2742, device='cuda:0', dtype=torch.float64)

In [ ]:
sim

tensor([[1.0000, 0.4456, 0.3992,  ..., 0.2085, 0.2712, 0.7207],
        [0.4456, 1.0000, 0.5436,  ..., 0.2421, 0.1923, 0.3144],
        [0.3992, 0.5436, 1.0000,  ..., 0.1547, 0.2400, 0.2860],
        ...,
        [0.2085, 0.2421, 0.1547,  ..., 1.0000, 0.3528, 0.2597],
        [0.2712, 0.1923, 0.2400,  ..., 0.3528, 1.0000, 0.3121],
        [0.7207, 0.3144, 0.2860,  ..., 0.2597, 0.3121, 1.0000]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
n_embeddings

15134

## Mais testes sintáticos

In [ ]:
n_embeddings = 9

tam = 3

lista_embeddings = torch.tensor([ [ [1,2,3], [4, 5, 6], [7, 8, 9] ],
                     [ [10,20,30], [40, 50, 60], [70, 80, 90] ],
                     [ [100,200,300], [400, 500, 600], [700, 800, 900] ]])

print(lista_embeddings)
#na vdd é lista de listas de listas!!!! pq cada embedding tem varios valores
# frase -> embedding -> valor

matriz = torch.zeros([n_embeddings, tam], dtype=torch.float64, device='cuda')
indice_matriz = 0
for i in range(len(lista_embeddings)):
  for j in range(len(lista_embeddings[i])):
    matriz[indice_matriz] = lista_embeddings[i][j]
    indice_matriz += 1

matriz

tensor([[[  1,   2,   3],
         [  4,   5,   6],
         [  7,   8,   9]],

        [[ 10,  20,  30],
         [ 40,  50,  60],
         [ 70,  80,  90]],

        [[100, 200, 300],
         [400, 500, 600],
         [700, 800, 900]]])


tensor([[  1.,   2.,   3.],
        [  4.,   5.,   6.],
        [  7.,   8.,   9.],
        [ 10.,  20.,  30.],
        [ 40.,  50.,  60.],
        [ 70.,  80.,  90.],
        [100., 200., 300.],
        [400., 500., 600.],
        [700., 800., 900.]], device='cuda:0', dtype=torch.float64)

In [ ]:
inicio_batch_i  = 3
fim_batch_i  = 6

matrizinha = matriz[inicio_batch_i: fim_batch_i]

matrizinha

tensor([[10., 20., 30.],
        [40., 50., 60.],
        [70., 80., 90.]], device='cuda:0', dtype=torch.float64)

In [ ]:
get_stopwords(stopwords_file)

{'a',
 'agora',
 'ainda',
 'algum',
 'alguma',
 'algumas',
 'alguns',
 'alguém',
 'ampla',
 'amplas',
 'amplo',
 'amplos',
 'ano',
 'anos',
 'ante',
 'antes',
 'ao',
 'aos',
 'apenas',
 'após',
 'aquela',
 'aquelas',
 'aquele',
 'aqueles',
 'aquilo',
 'as',
 'através',
 'até',
 'bem',
 'cada',
 'cento',
 'coisa',
 'coisas',
 'com',
 'como',
 'contos',
 'contra',
 'contudo',
 'da',
 'daquele',
 'daqueles',
 'das',
 'de',
 'dela',
 'delas',
 'dele',
 'deles',
 'depois',
 'dessa',
 'dessas',
 'desse',
 'desses',
 'desta',
 'destas',
 'deste',
 'destes',
 'deve',
 'devem',
 'devendo',
 'dever',
 'deveria',
 'deveriam',
 'deverá',
 'deverão',
 'devia',
 'deviam',
 'dia',
 'disse',
 'disso',
 'disto',
 'dito',
 'diz',
 'dizem',
 'do',
 'dois',
 'dos',
 'durante',
 'e',
 'ela',
 'elas',
 'ele',
 'eles',
 'em',
 'enquanto',
 'entre',
 'era',
 'essa',
 'essas',
 'esse',
 'esses',
 'esta',
 'estado',
 'estamos',
 'estas',
 'estava',
 'estavam',
 'este',
 'estes',
 'estou',
 'está',
 'estávamos',

In [ ]:
frase1 = "ontem choveu muito"
frase2= "amanhã vai chover"

In [ ]:
tokens1, embeddings1 = tokens_embeddings(frase1, tokenizer, model)

In [ ]:
tokens1

['[CLS]', 'on', '##tem', 'cho', '##veu', 'muito', '[SEP]']

In [ ]:
tokens2, embeddings2 = tokens_embeddings(frase2, tokenizer, model)

In [ ]:
tokens2

['[CLS]', 'aman', '##ha', 'vai', 'cho', '##ver', '[SEP]']

In [ ]:
s = similaridade(embeddings1[3], embeddings2[4])

In [ ]:
s

0.8764368295669556

In [ ]:
s.item()

0.8764368295669556

In [ ]:
len(embeddings2) - len(tokens2)

0

In [ ]:
emb1.dim()

1

In [ ]:
a = torch.tensor([ 4.4662e-01, -5.1519e-01,  4.0550e-01,  1.3108e-01,  5.5277e-01,
         4.2338e-01,  4.3060e-02])

a

tensor([ 0.4466, -0.5152,  0.4055,  0.1311,  0.5528,  0.4234,  0.0431])

In [ ]:
b = torch.tensor([ 5.4662e-01, -4.1519e-01,  5.0550e-01,  2.3108e-01,  6.5277e-01,
         5.2338e-01,  5.3060e-02])
b

tensor([ 0.5466, -0.4152,  0.5055,  0.2311,  0.6528,  0.5234,  0.0531])

In [ ]:
a.dim()

1

In [ ]:
a.shape

torch.Size([7])

In [ ]:
b.dim

In [ ]:
lista = ["a", "b", "c"]

In [ ]:
lista[0]

'a'

In [ ]:
lista[0] = "k"

In [ ]:
lista

['k', 'b', 'c']

In [ ]:
lista  = [["a", "b", "c", "d", "e", "f"], ["g", "a", "d", "h", "i", "j"], ["k", "l", "m", "h", "a", "b"]]



In [ ]:
lista[0][0] = "a_1"
lista[1][1] = "a_1"

lista[0][3] = "d_1"
lista[1][2] = "d_1"

lista


[['a_1', 'b', 'c', 'd_1', 'e', 'f'],
 ['g', 'a_1', 'd_1', 'h', 'i', 'j'],
 ['k', 'l', 'm', 'h', 'a', 'b']]

In [ ]:
#concatenando horizontalmente

In [ ]:
matriz = torch.zeros([6, 10], dtype=torch.float64, device='cuda')


matriz

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0',
       dtype=torch.float64)

In [ ]:
batch1 = torch.zeros([6,3], dtype=torch.float64, device='cuda' )
batch2 = torch.zeros([6,3], dtype=torch.float64, device='cuda' )
batch3 = torch.zeros([6,4], dtype=torch.float64, device='cuda' )

In [ ]:
batch1[0] = torch.tensor([1, 2, 3])
batch1[1] = torch.tensor([4, 5, 6])
batch1[2] = torch.tensor([7, 8, 9])
batch1[3] = torch.tensor([10, 11, 12])
batch1[4] = torch.tensor([13, 14, 15])
batch1[5] = torch.tensor([16, 17, 18])

batch1

tensor([[ 1.,  2.,  3.],
        [ 4.,  5.,  6.],
        [ 7.,  8.,  9.],
        [10., 11., 12.],
        [13., 14., 15.],
        [16., 17., 18.]], device='cuda:0', dtype=torch.float64)

In [ ]:
batch2[0] = torch.tensor([10, 20, 30])
batch2[1] = torch.tensor([40, 50, 60])
batch2[2] = torch.tensor([70, 80, 90])
batch2[3] = torch.tensor([100, 110, 120])
batch2[4] = torch.tensor([130, 140, 150])
batch2[5] = torch.tensor([160, 170, 180])

batch2

tensor([[ 10.,  20.,  30.],
        [ 40.,  50.,  60.],
        [ 70.,  80.,  90.],
        [100., 110., 120.],
        [130., 140., 150.],
        [160., 170., 180.]], device='cuda:0', dtype=torch.float64)

In [ ]:
batch3[0] = torch.tensor([100, 200, 300, 3000])
batch3[1] = torch.tensor([400, 500, 600, 6000])
batch3[2] = torch.tensor([700, 800, 900, 9000])
batch3[3] = torch.tensor([1000, 1100, 1200, 12000])
batch3[4] = torch.tensor([1300, 1400, 1500, 15000])
batch3[5] = torch.tensor([1600, 1700, 1800, 18000])

batch3

tensor([[  100.,   200.,   300.,  3000.],
        [  400.,   500.,   600.,  6000.],
        [  700.,   800.,   900.,  9000.],
        [ 1000.,  1100.,  1200., 12000.],
        [ 1300.,  1400.,  1500., 15000.],
        [ 1600.,  1700.,  1800., 18000.]], device='cuda:0',
       dtype=torch.float64)

In [ ]:
matriz_nova = torch.hstack([batch1, batch2, batch3])
matriz_nova

tensor([[  1.,   2.,   3.,  10.,  20.,  30.,   0.,   0.,   0.,   0.],
        [  4.,   5.,   6.,  40.,  50.,  60.,   0.,   0.,   0.,   0.],
        [  7.,   8.,   9.,  70.,  80.,  90.,   0.,   0.,   0.,   0.],
        [ 10.,  11.,  12., 100., 110., 120.,   0.,   0.,   0.,   0.],
        [ 13.,  14.,  15., 130., 140., 150.,   0.,   0.,   0.,   0.],
        [ 16.,  17.,  18., 160., 170., 180.,   0.,   0.,   0.,   0.]],
       device='cuda:0', dtype=torch.float64)

##  Testando aplicação de uma função em cada elemento do tensor, com GPU


In [ ]:
matriz = torch.zeros([10, 15], dtype=torch.float64, device='cuda')

vetor  = np.random.rand(10,15) * 10

vetor

vetor_torch = torch.tensor(vetor, device = 'cuda')

matriz = vetor_torch

matriz

tensor([[1.9733, 5.6260, 1.1761, 8.3875, 9.2377, 5.2749, 1.6065, 1.5295, 9.1281,
         0.3241, 4.5726, 9.4551, 1.9384, 3.2425, 6.6121],
        [0.1644, 1.3169, 4.1657, 3.3146, 0.0220, 9.4950, 9.9741, 3.5093, 6.8303,
         7.8138, 4.5273, 5.9406, 1.9610, 7.9915, 2.0310],
        [5.2080, 9.9218, 6.6954, 7.7449, 0.4624, 1.6018, 4.0517, 9.6747, 5.9728,
         7.6970, 4.4859, 3.3336, 7.7434, 0.5149, 2.4276],
        [0.1766, 8.2320, 4.4818, 7.3779, 1.4429, 5.0420, 0.6852, 6.3474, 9.6864,
         8.4581, 4.5666, 1.0852, 7.2657, 3.6694, 7.5912],
        [0.1376, 9.2871, 2.7879, 2.3188, 6.3676, 5.2841, 3.6979, 7.1524, 5.4104,
         1.8878, 5.3060, 0.3485, 6.3839, 2.6925, 3.2816],
        [4.7237, 9.6911, 6.2652, 3.6757, 6.7079, 7.6255, 7.0051, 3.1751, 8.5233,
         9.5598, 5.8869, 2.3690, 8.9037, 4.3969, 4.8035],
        [0.6423, 5.2547, 4.3207, 0.8255, 2.0041, 2.1070, 7.3870, 5.2037, 0.1629,
         1.7910, 6.9834, 0.1088, 5.7128, 8.9706, 0.8579],
        [9.0549, 3.6500, 4.

In [ ]:
matriz = matriz.int()


matriz

tensor([[1, 5, 1, 8, 9, 5, 1, 1, 9, 0, 4, 9, 1, 3, 6],
        [0, 1, 4, 3, 0, 9, 9, 3, 6, 7, 4, 5, 1, 7, 2],
        [5, 9, 6, 7, 0, 1, 4, 9, 5, 7, 4, 3, 7, 0, 2],
        [0, 8, 4, 7, 1, 5, 0, 6, 9, 8, 4, 1, 7, 3, 7],
        [0, 9, 2, 2, 6, 5, 3, 7, 5, 1, 5, 0, 6, 2, 3],
        [4, 9, 6, 3, 6, 7, 7, 3, 8, 9, 5, 2, 8, 4, 4],
        [0, 5, 4, 0, 2, 2, 7, 5, 0, 1, 6, 0, 5, 8, 0],
        [9, 3, 4, 3, 9, 0, 7, 8, 0, 2, 7, 4, 2, 5, 6],
        [9, 7, 8, 5, 4, 4, 0, 3, 4, 9, 0, 6, 3, 0, 9],
        [1, 6, 1, 5, 2, 7, 3, 9, 4, 6, 2, 5, 7, 7, 6]], device='cuda:0',
       dtype=torch.int32)

In [ ]:
#voltando pra float
matriz = matriz.float()

matriz

tensor([[ 3., 13.,  4.,  9., 13., 11., 15., 19., 15., 10.,  7.,  9.,  4.,  0.,
         19.],
        [15.,  4.,  0., 16., 14., 16.,  1.,  7., 15., 13.,  1.,  3., 10., 15.,
          9.],
        [17.,  7.,  4.,  3.,  4.,  3.,  5., 17., 15., 10., 11.,  8.,  1., 13.,
          6.],
        [ 1., 10., 10., 15., 15., 11., 17., 19., 13., 13.,  0.,  0., 19.,  6.,
         17.],
        [12.,  8.,  8., 17.,  1., 12.,  2.,  4.,  9.,  3.,  3.,  1., 18., 11.,
          6.],
        [16.,  9., 13.,  7., 16.,  7.,  4.,  4., 16.,  8.,  0., 15., 13., 18.,
          6.],
        [19., 12.,  5.,  5.,  6.,  8., 14., 12.,  1.,  3.,  1., 14.,  2.,  4.,
          7.],
        [ 4., 17., 16.,  7., 16.,  4., 18., 10.,  0.,  9.,  1.,  9., 17., 17.,
          1.],
        [ 2.,  3., 15.,  0., 14.,  4.,  9.,  2., 14.,  2.,  6.,  5.,  9.,  4.,
          9.],
        [10.,  5.,  8.,  7., 10., 18., 18.,  2., 19.,  7.,  7., 19.,  4., 14.,
         11.]], device='cuda:0')

In [ ]:
matriz.dtype

torch.float32

In [ ]:
#somando matrizes grandes, sendo uma delas um valor constante

matriz = torch.zeros([10000, 768], dtype=torch.float64, device='cuda')

vetor  = np.random.rand(10000,768) * 10

vetor_torch = torch.tensor(vetor, device = 'cuda')

matriz = vetor_torch

matriz


tensor([[7.3575, 8.1591, 7.7735,  ..., 9.0654, 5.3070, 9.4392],
        [4.1177, 3.9167, 2.6582,  ..., 4.5208, 7.9849, 4.4320],
        [2.5509, 0.3211, 5.6026,  ..., 6.1277, 1.8436, 5.6317],
        ...,
        [7.2648, 2.2034, 5.2150,  ..., 5.9602, 9.9951, 3.4040],
        [5.7059, 6.0927, 4.7288,  ..., 2.4908, 1.9207, 1.2933],
        [0.0362, 4.7634, 9.2368,  ..., 7.3793, 3.4682, 1.8563]],
       device='cuda:0', dtype=torch.float64)

In [ ]:


k = 0.25
matriz2 = torch.ones([10000, 768], dtype=torch.float64, device='cuda')

#vetor_torch = torch.tensor([[0.25 for i in range(768) ] for j in range(10000)], device = 'cuda')

matriz2=matriz2*k
matriz2

tensor([[0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        ...,
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
soma = matriz + matriz2

In [ ]:
#foi rapido !

In [ ]:
soma

tensor([[ 7.6075,  8.4091,  8.0235,  ...,  9.3154,  5.5570,  9.6892],
        [ 4.3677,  4.1667,  2.9082,  ...,  4.7708,  8.2349,  4.6820],
        [ 2.8009,  0.5711,  5.8526,  ...,  6.3777,  2.0936,  5.8817],
        ...,
        [ 7.5148,  2.4534,  5.4650,  ...,  6.2102, 10.2451,  3.6540],
        [ 5.9559,  6.3427,  4.9788,  ...,  2.7408,  2.1707,  1.5433],
        [ 0.2862,  5.0134,  9.4868,  ...,  7.6293,  3.7182,  2.1063]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
matriz_esparsa = soma.int()

matriz_esparsa

tensor([[ 7,  8,  8,  ...,  9,  5,  9],
        [ 4,  4,  2,  ...,  4,  8,  4],
        [ 2,  0,  5,  ...,  6,  2,  5],
        ...,
        [ 7,  2,  5,  ...,  6, 10,  3],
        [ 5,  6,  4,  ...,  2,  2,  1],
        [ 0,  5,  9,  ...,  7,  3,  2]], device='cuda:0', dtype=torch.int32)

In [ ]:
uns = torch.ones(10000,768, device = 'cuda')

In [ ]:
uns

tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]], device='cuda:0')

In [ ]:
uns * 0.24

tensor([[0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400],
        [0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400],
        [0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400],
        ...,
        [0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400],
        [0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400],
        [0.2400, 0.2400, 0.2400,  ..., 0.2400, 0.2400, 0.2400]],
       device='cuda:0')

In [ ]:
uns + matriz

tensor([[ 6.0330,  3.8026, 10.6690,  ..., 17.9405, 12.0654,  4.3750],
        [ 7.9004, 11.0332, 18.1326,  ...,  8.2757,  2.6242, 19.8667],
        [ 4.8345, 11.7476, 17.9256,  ..., 18.6965,  2.8405,  8.8665],
        ...,
        [ 6.0069,  4.0467, 18.0391,  ..., 16.7974,  3.1841, 18.5700],
        [10.8079, 19.8793, 18.2647,  ..., 20.6970, 14.9282,  8.3147],
        [13.3475,  6.2977,  9.5161,  ...,  2.7626, 18.1066,  7.0346]],
       device='cuda:0', dtype=torch.float64)

## Multiplicando termo a termo cada vetor linha por [1, 2, 3, 4, 5....]

In [ ]:
#ajustando a matriz esparsa pra so ter 0 e 1

In [ ]:
matriz_esparsa.float()

tensor([[ 7.,  8.,  8.,  ...,  9.,  5.,  9.],
        [ 4.,  4.,  2.,  ...,  4.,  8.,  4.],
        [ 2.,  0.,  5.,  ...,  6.,  2.,  5.],
        ...,
        [ 7.,  2.,  5.,  ...,  6., 10.,  3.],
        [ 5.,  6.,  4.,  ...,  2.,  2.,  1.],
        [ 0.,  5.,  9.,  ...,  7.,  3.,  2.]], device='cuda:0')

In [ ]:
matriz_esparsa = matriz_esparsa/10

In [ ]:
matriz_esparsa

tensor([[0.7000, 0.8000, 0.8000,  ..., 0.9000, 0.5000, 0.9000],
        [0.4000, 0.4000, 0.2000,  ..., 0.4000, 0.8000, 0.4000],
        [0.2000, 0.0000, 0.5000,  ..., 0.6000, 0.2000, 0.5000],
        ...,
        [0.7000, 0.2000, 0.5000,  ..., 0.6000, 1.0000, 0.3000],
        [0.5000, 0.6000, 0.4000,  ..., 0.2000, 0.2000, 0.1000],
        [0.0000, 0.5000, 0.9000,  ..., 0.7000, 0.3000, 0.2000]],
       device='cuda:0')

In [ ]:
k = 0.25
matriz2 = torch.ones([10000, 768], dtype=torch.float64, device='cuda')

#vetor_torch = torch.tensor([[0.25 for i in range(768) ] for j in range(10000)], device = 'cuda')

matriz2=matriz2*k
matriz2

tensor([[0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        ...,
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500,  ..., 0.2500, 0.2500, 0.2500]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
matriz_esparsa = matriz_esparsa + matriz2

In [ ]:
matriz_esparsa

tensor([[0.9500, 1.0500, 1.0500,  ..., 1.1500, 0.7500, 1.1500],
        [0.6500, 0.6500, 0.4500,  ..., 0.6500, 1.0500, 0.6500],
        [0.4500, 0.2500, 0.7500,  ..., 0.8500, 0.4500, 0.7500],
        ...,
        [0.9500, 0.4500, 0.7500,  ..., 0.8500, 1.2500, 0.5500],
        [0.7500, 0.8500, 0.6500,  ..., 0.4500, 0.4500, 0.3500],
        [0.2500, 0.7500, 1.1500,  ..., 0.9500, 0.5500, 0.4500]],
       device='cuda:0', dtype=torch.float64)

In [ ]:
matriz_esparsa = matriz_esparsa.int()

tensor([[0, 1, 1,  ..., 1, 0, 1],
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0]], device='cuda:0', dtype=torch.int32)

In [ ]:
#OBS: diferente do que vai acontecer na prática, aqui varios vetores começam com 0...
#mas vamos seguir

In [ ]:
import copy
matriz_backup = copy.deepcopy(matriz_esparsa)

In [ ]:
matriz_backup

tensor([[0, 1, 1,  ..., 1, 0, 1],
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 1, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 1,  ..., 0, 0, 0]], device='cuda:0', dtype=torch.int32)

In [ ]:
sequencia = torch.zeros(1,768, device= 'cuda')

num = 1
for i in range(len(sequencia[0])):
  sequencia[0][i] = num
  num +=1

#sequencia[0][3] = 6
sequencia


tensor([[  1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,  11.,  12.,
          13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,  22.,  23.,  24.,
          25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,  33.,  34.,  35.,  36.,
          37.,  38.,  39.,  40.,  41.,  42.,  43.,  44.,  45.,  46.,  47.,  48.,
          49.,  50.,  51.,  52.,  53.,  54.,  55.,  56.,  57.,  58.,  59.,  60.,
          61.,  62.,  63.,  64.,  65.,  66.,  67.,  68.,  69.,  70.,  71.,  72.,
          73.,  74.,  75.,  76.,  77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,
          85.,  86.,  87.,  88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,
          97.,  98.,  99., 100., 101., 102., 103., 104., 105., 106., 107., 108.,
         109., 110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
         121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131., 132.,
         133., 134., 135., 136., 137., 138., 139., 140., 141., 142., 143., 144.,
         145., 146., 147., 1

In [ ]:
matriz_esparsa * sequencia

tensor([[  0.,   2.,   3.,  ..., 766.,   0., 768.],
        [  0.,   0.,   0.,  ...,   0., 767.,   0.],
        [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
        ...,
        [  0.,   0.,   0.,  ...,   0., 767.,   0.],
        [  0.,   0.,   0.,  ...,   0.,   0.,   0.],
        [  0.,   0.,   3.,  ...,   0.,   0.,   0.]], device='cuda:0')

## Eliminando Zeros com numpy  -SINTAXE & PERFORMANCE
### Para usar com a matriz binária * [1 2 3 4 5 6 7 8 9 ...]

 Update! Não precisa usar numpy! só usar a MESMA notação [ X[X != 0] for X in Tensor]  com o torch!!

In [ ]:
# @title
#"single" test
X = np.random.randn(1, 1000000)
X[np.abs(X)< .22]= 0 # alguns zeros

print(X[0][:30])
X = X[X !=0]
print("\n\n\n")
print(X[:30])
print(X[5:10])
print(X[10:15])

[ 0.         -0.34543331 -0.33978351 -2.09453221  0.6590586  -1.4054663
 -0.40173629  0.7468376   0.38834137 -0.68734926  0.          0.
 -1.30258299  0.47214621  0.          1.77760835 -0.49259363 -0.72089694
  0.82064779  0.          1.05628438  0.6347628  -1.32025255 -1.35560256
  1.10426668 -0.35774033 -0.33349953  0.         -0.60231772 -0.7149956 ]




[-0.34543331 -0.33978351 -2.09453221  0.6590586  -1.4054663  -0.40173629
  0.7468376   0.38834137 -0.68734926 -1.30258299  0.47214621  1.77760835
 -0.49259363 -0.72089694  0.82064779  1.05628438  0.6347628  -1.32025255
 -1.35560256  1.10426668 -0.35774033 -0.33349953 -0.60231772 -0.7149956
  0.9477042   0.94100959  1.13788665  2.23567546 -0.39498614  0.31923932]
[-0.40173629  0.7468376   0.38834137 -0.68734926 -1.30258299]
[ 0.47214621  1.77760835 -0.49259363 -0.72089694  0.82064779]


In [ ]:
# @title
#testando direto com matriz

A = np.random.randn(200, 600) #120000 valores
A[np.abs(A)< .22]= 0 # alguns zeros

print(A[0][:30])
print("\n\n")
print(A[4][:30])

A = A[A !=0]

print("Depois de tirar os zeros")


print("shape: ", A.shape)
#virou um vetor só. Mas como fazer o parsing? Um jeito seria
#marcar a diagonal com -1. Sempre olharemos então À frente da diagonal
#isso é "Justo"? Ver definição da página 47.
#De toda forma não tem mt como parsear pq eu não sei onde
#acaba cada linha

print(A[:30])


[-1.11306272  0.41865839  0.86038122  1.2124047   0.77735112  1.07107055
  0.          0.37289093 -0.62302872 -0.43647757  0.          0.25186367
 -0.80739705  0.          0.         -1.44513763 -0.77533244  0.69309951
  0.         -1.56626311  0.64091527  1.58097692 -0.47706631  2.52588799
 -2.01181932  1.32857009 -0.27910036 -1.93865811  0.43641441  1.63734609]



[ 0.         -1.54654342  0.35905737  0.72144375 -1.91139158 -0.56192152
  0.61181464 -0.58374309  0.          0.          1.88616574  0.
  0.          0.7377797  -1.4572653   0.         -1.55412185  0.97649685
  0.46001243 -0.58442579  2.36123022 -0.50282215  0.82641808 -0.45896541
  0.54267292  1.38319501 -0.73565439  0.          1.54171417  0.82332649]
Depois de tirar os zeros
shape:  (99229,)
[-1.11306272  0.41865839  0.86038122  1.2124047   0.77735112  1.07107055
  0.37289093 -0.62302872 -0.43647757  0.25186367 -0.80739705 -1.44513763
 -0.77533244  0.69309951 -1.56626311  0.64091527  1.58097692 -0.47706631
  2.52588799

In [ ]:
# @title
#testando como (provavelmente) será feito: matrizona torch, convertendo
#pra numpy (portanto usando CPU por enquanto) e pegando cada linha


n_linhas = 10000

A = np.random.randn(n_linhas, 50000)
A[np.abs(A)< .22]= 0 # alguns zeros

T = torch.tensor(A, device = 'cuda')
T = T*100
T = T.int()



In [ ]:
# @title
%%time
T = T.cpu()

B = np.array(T)

#testando
#copia = [linha[0] for linha in B]

#print(copia)

#print(B)

listas = [c[c !=0] for c in B]

#print(len(listas[0]))
#print(len(listas[1]))
#print(len(listas[2]))

#print("\n\n\n")

#print(listas[:3])

CPU times: user 3.01 s, sys: 1.73 s, total: 4.74 s
Wall time: 4.74 s


 Resultado do teste de performance (tirando os zeros da matriz):
 QUATRO.74 SEGUNDOS para 500 MILHÕES DE VALORES (10.000 X 50.000)

Isso daria 14,22 segundos para 12GB de dados (1,5B valores de 8 Bytes).

## Eliminando zeros com torch
### Teste de Performance

In [ ]:
n_linhas = 10000

A = np.random.randn(n_linhas, 5000)
A[np.abs(A)< .22]= 0 # alguns zeros

T = torch.tensor(A, device = 'cuda')
T = T*100
T = T.int()

#T = T.cpu()

#B = np.array(T)

#testando
#copia = [linha[0] for linha in B]

#print(copia)

#print(B)
print(T[0])

%time listas = [c[c !=0] for c in T]

print(listas[:30])

tensor([-79,   0, 173,  ..., -67, -47,  53], device='cuda:0',
       dtype=torch.int32)
CPU times: user 743 ms, sys: 18.2 ms, total: 761 ms
Wall time: 761 ms
[tensor([-79, 173, 159,  ..., -67, -47,  53], device='cuda:0',
       dtype=torch.int32), tensor([-166, -100,   57,  ...,  109,  122,  117], device='cuda:0',
       dtype=torch.int32), tensor([168,  37, -34,  ..., -72, -90,  54], device='cuda:0',
       dtype=torch.int32), tensor([ 63,  35, -33,  ...,  88, 199, -91], device='cuda:0',
       dtype=torch.int32), tensor([-38, -36, -78,  ..., 136, -44, -97], device='cuda:0',
       dtype=torch.int32), tensor([-186, -308, -148,  ..., -124,   64, -102], device='cuda:0',
       dtype=torch.int32), tensor([-41, -77,  61,  ..., 125,  52, 135], device='cuda:0',
       dtype=torch.int32), tensor([  80,  103, -131,  ...,   87, -111,   71], device='cuda:0',
       dtype=torch.int32), tensor([ -57, -106,   81,  ...,   57,  191, -100], device='cuda:0',
       dtype=torch.int32), tensor([ 161,  -

###Resultado do Teste de performance:
743 milissegundos para 50 milhões de valores (10000 X 5000)

22.3 segundos para 1.5B (12 GB de dados se for 8 bytes) de valores se for linear

excelente!!

In [ ]:
print(listas[0][:30])

tensor([ 169,  104,   68,  -82,  -62,  -99,  123, -152, -316,  -46,   64,  -65,
         -63,   82,  118,   79, -146,  102,   87,  161,  -32,   84, -151,  -58,
         122,  -32,   78, -176, -208,  -69], device='cuda:0',
       dtype=torch.int32)


In [ ]:
## Resumo: DÁ PRA FAZER!!!

## Testando de uma vez o processamento novo
### Somar 1-threshold na matriz de similaridades, converter pra int, multiplicar por [1 2 3 4 5 6 ...] linha a linha, tirar os zeros

In [ ]:
threshold = 0.75

n_linhas = 1000
n_col = 500

A = np.random.randn(n_linhas, n_col)
A[np.abs(A) > 1]= 0.78 #simulando a matriz de similaridades
A = abs(A)

#print(A)

T = torch.tensor(A, device = 'cuda')

matriz_p_soma = torch.ones(n_linhas, n_col, device = 'cuda') * (1-threshold)

#print(matriz_p_soma)

T = T + matriz_p_soma
#print(T)

T = T.int()

#print(T)


#fazendo a sequencia
#e sim, começa do 1!!!! Tomar cuidado!

#definir n_col!!! tem que ser igual o numero de tokens!
sequencia = torch.zeros(1,n_col, device= 'cuda')

num = 1
for i in range(len(sequencia[0])):
  sequencia[0][i] = num
  num +=1

#print(sequencia)

T = T*sequencia

#print(T)

listas =[(l [l !=0])-1 for l in T]
#obs: aqui estamos quase dobrando a memória!
#obs2: as listas tem tamanhos diferentes

#print("\n\n")
print(listas[:30])

#print("\n\n")

[tensor([  1.,   4.,   5.,   6.,   8.,  14.,  15.,  17.,  18.,  19.,  21.,  24.,
         25.,  26.,  27.,  28.,  29.,  33.,  38.,  40.,  41.,  42.,  43.,  45.,
         47.,  48.,  49.,  51.,  52.,  53.,  55.,  59.,  60.,  62.,  63.,  64.,
         65.,  66.,  68.,  72.,  73.,  75.,  80.,  85.,  86.,  93.,  95.,  97.,
         98.,  99., 100., 101., 102., 103., 105., 106., 107., 108., 112., 113.,
        114., 117., 118., 121., 123., 124., 125., 126., 130., 131., 133., 134.,
        135., 136., 137., 138., 139., 147., 148., 149., 151., 159., 160., 161.,
        162., 163., 165., 166., 168., 170., 172., 173., 176., 177., 180., 183.,
        189., 193., 194., 195., 197., 199., 200., 202., 205., 209., 210., 216.,
        217., 219., 220., 221., 222., 223., 225., 226., 233., 234., 236., 243.,
        245., 247., 248., 250., 251., 252., 253., 254., 255., 257., 259., 260.,
        261., 262., 263., 265., 267., 269., 271., 273., 279., 284., 286., 291.,
        295., 297., 301., 302., 305., 3

Mais testes

In [ ]:
threshold = 0.75

n_linhas = 1000
n_col = 500

A = np.random.randn(n_linhas, n_col)
A[np.abs(A) > 1]= 0.78 #simulando a matriz de similaridades
A = abs(A)

print(A)
print("\n\n")

T = torch.tensor(A, device = 'cuda')

vetor_p_soma = torch.ones(1, n_col, device = 'cuda') * (1-threshold)

print(vetor_p_soma)
print("\n\n")

T = T + vetor_p_soma
print(T)



[[0.12668109 0.17859124 0.78       ... 0.43920176 0.58695928 0.78      ]
 [0.98671347 0.21800194 0.6138998  ... 0.93620303 0.59553385 0.78      ]
 [0.46149792 0.78       0.78       ... 0.43619682 0.29159144 0.36787784]
 ...
 [0.78       0.1142515  0.78       ... 0.78       0.12109322 0.22456633]
 [0.25404847 0.55703891 0.89605077 ... 0.78       0.72165141 0.08114286]
 [0.78       0.78       0.37453006 ... 0.78       0.06764139 0.40843357]]



tensor([[0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500,
         0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.

In [ ]:
a = torch.ones(1,200, device='cuda')
b = a - 1
b

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0.]], device='cuda:0')

In [ ]:

n_tokens = 60

conjuntos = [] #componente conexo ao qual cada token pertence
                        #considerando o grafo formado pela matriz binária
                        #de similaridades


for i in range(n_tokens):
  conjuntos.append(i)
#originalmente, cada token pertence ao seu
#proprio componente conexo

conjuntos_tokens = torch.tensor(conjuntos, device='cuda')

conjuntos = []

conjuntos_tokens


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59], device='cuda:0')

In [ ]:
#pra parte de redesignação de tokens em processa_janela_bert_fast

conjuntos_atuais = torch.tensor([6,13,4,7,8,6,4,5], device = 'cuda')

menor = torch.min(conjuntos_atuais, dim = 0)
menor_valor = menor.values
indice_menor = int(menor.indices) + 60

listona = [i for i in range(200)]

print(listona[indice_menor])
print(str(menor_valor))

#indice_menor = conjuntos_atuais.index(menor)

#indice_menor

62
tensor(4, device='cuda:0')


In [ ]:
lista_tokens = [["sphinx", "of", "black", "quartz", "judge", "my", "vow"], ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"],
                ["essa", "frase", "é", "falsa"]]


listona_tokens = []
for frase in lista_tokens:
  listona_tokens.extend(frase)


listona_tokens

['sphinx',
 'of',
 'black',
 'quartz',
 'judge',
 'my',
 'vow',
 'the',
 'quick',
 'brown',
 'fox',
 'jumps',
 'over',
 'the',
 'lazy',
 'dog',
 'essa',
 'frase',
 'é',
 'falsa']

### Hstack


In [ ]:
n_tokens = 30
tam = 90

matriz = torch.zeros((n_tokens, tam), dtype=torch.float64, device='cuda')

matriz.shape

torch.Size([30, 90])

In [ ]:
matriz2 = torch.ones((n_tokens, tam), dtype=torch.float64, device='cuda')
matriz2

tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]], device='cuda:0', dtype=torch.float64)

In [ ]:
ultima = (torch.ones((n_tokens, tam), dtype=torch.float64, device='cuda')) *2

ultima

tensor([[2., 2., 2.,  ..., 2., 2., 2.],
        [2., 2., 2.,  ..., 2., 2., 2.],
        [2., 2., 2.,  ..., 2., 2., 2.],
        ...,
        [2., 2., 2.,  ..., 2., 2., 2.],
        [2., 2., 2.,  ..., 2., 2., 2.],
        [2., 2., 2.,  ..., 2., 2., 2.]], device='cuda:0', dtype=torch.float64)

In [ ]:
lista_matrizes = []
lista_matrizes.append(matriz)
lista_matrizes.append(matriz2)
lista_matrizes.append(ultima)

matriz3 = torch.hstack(lista_matrizes)

print(matriz3)

print(matriz3[0][50]) #0

print(matriz3[0][95]) #1

print(matriz3[0][200]) #2

tensor([[0., 0., 0.,  ..., 2., 2., 2.],
        [0., 0., 0.,  ..., 2., 2., 2.],
        [0., 0., 0.,  ..., 2., 2., 2.],
        ...,
        [0., 0., 0.,  ..., 2., 2., 2.],
        [0., 0., 0.,  ..., 2., 2., 2.],
        [0., 0., 0.,  ..., 2., 2., 2.]], device='cuda:0', dtype=torch.float64)
tensor(0., device='cuda:0', dtype=torch.float64)
tensor(1., device='cuda:0', dtype=torch.float64)
tensor(2., device='cuda:0', dtype=torch.float64)


### Testando embeddings de 4 bytes

In [ ]:
frase = "nao sei porque que nao esta tando certo"
tokens, embeddings = tokens_embeddings(frase, tokenizer, model)

#print(embeddings[0][0])
embeddings.dtype

torch.float32

In [ ]:
#Funcionou! é float 32

### Mais testes sintáticos

In [ ]:
similaridades = torch.ones([20,30], device = 'cuda')


n_linhas,n_col = similaridades.size()
print(type(n_linhas))
print(n_col)

<class 'int'>
30


In [ ]:
frase = "nao sei porque que nao esta tando certo"
tokens, embeddings = tokens_embeddings(frase, tokenizer, model)

print(tokens)

['[CLS]', 'na', '##o', 'sei', 'porque', 'que', 'na', '##o', 'esta', 'tan', '##do', 'certo', '[SEP]']


In [ ]:
5000 * 25 * 5000 * 25* 8

125000000000

In [ ]:
''' [[1,  0.5, 0.4, 0.8, 0.9, 0.1, 0.85]                 + [0,25] #1 - limiar
 [0.5, 1,   0.6,   0.9,   0.2,  0.04,   0.77 ]
 ]

matriz2 = matriz.int()

[[1,  0, 0, 1, 1, 0, 1]                    * [1, 2, 3, 4, 5, 6, 7]   = [1, 0,0,4,5, 0, 7]
 [0, 1,   0,   1,   0,  0, 1 ]
 ]

 [1, 0,0,4,5, 0, 7]


 [1,4,5, 7] #tokens 1, 4, 5, e 7 são similares ao token 1

 [0, 3, 4, 6] #posição na lista de todos os tokens dos tokens similares




t0 :[0, 3, 4, 6]

t1: [1, 5, 15, 21]

t2: [2, 6, 8]

t3: [3, 9]

...

T0 = {0,2, 3, 4, 6, 8, 9}

T1 = {1, 5, 15, 21}

componentes_tokens = [0,1,0,0,0,1,0,6,0,0...]

" t1 t2 ... sars \n
coronavirus t30 t31"

->

#'''#t1 t2 .... sars_3
#sars_3  t30 t31'''


'''

In [ ]:
torch.cuda.empty_cache()